# Fabric Estate Scan

**One notebook. All ten delivery scripts. Complete read-only assessment of a Microsoft Fabric tenant.**

Everything runs inside Fabric and writes results back to the attached Lakehouse as `estate_*` tables.

## Coverage

### Part One — metadata only, no data access

| § | Section | Script | Session |
|---|---|---|---|
| 1 | Tenant inventory — capacities, workspaces, items, OneLake tables, shortcuts | `01` | S01 |
| 2 | Scanner metadata — models, **relationships**, RLS, measures, M queries, lineage | `07` | S01 |
| 3 | **Delta profiling** — schema, files, size, **rebuild vs MERGE** | `03` | S01 |
| 4 | Governance — domains, Git, deployment pipelines, roles, schedules, tenant settings | `07` | S01 |
| 4b | **Capacity metrics** — utilisation, throttling, top consumers | `06` | S01 |
| 5 | Item definitions — pipeline activities with source/sink/dependsOn | `01` | S01 |
| 6 | **SQL endpoint** — views vs persisted tables, view chain depth | `02` | S01 |
| 7 | Estate graph — nodes, edges, Graphviz DOT | — | S01 |
| 8 | Findings | — | S01 |

### Part Two — reads row-level data, off by default

| § | Section | Script | Session |
|---|---|---|---|
| 9 | Six quality dimensions | `04` | S03 |
| 10 | Study / Site / Subject reconciliation | `05` | S02 |
| 11 | Great Expectations suite | `10` | S03 |
| 12 | Candidate semantic model (TMDL) | `08` | S05 |
| 13 | Fabric IQ ontology binding spec | `09` | S05 |

> **`READ_DATA_CONTENT` defaults to `False`.** Part One runs alone under a light approval. Sections 9–11 open clinical data — a different permission posture and potentially PHI. Set the flag only when data access has been agreed.

## The three sections that justify the exercise

**§3 Delta profiling** reads the transaction log. `DESCRIBE HISTORY` tells you whether a table is incrementally `MERGE`d or fully rewritten every run. A Gold table can be persisted *and* rebuilt nightly — costing nearly what a view costs while looking correct on an architecture diagram. No REST API exposes this.

**§6 SQL endpoint** is the only place views are visible. Views have no Delta log and are not returned by the tables API, so a Warehouse of chained views produces silence from §1 and §3 — silence you would misread as "no Gold objects." Chain depth ≥ 2 means every read recomputes the layers beneath it.

**§10 Reconciliation** is the strongest accuracy evidence available without a gold standard. Two independent systems agreeing means something that one system being internally consistent does not.

## Prerequisites

| Need | Consequence if missing |
|---|---|
| **Fabric Administrator** | §2 skipped — no relationships, RLS, lineage or model schema |
| Tenant setting: *Enhance admin APIs responses with detailed metadata* | §2 returns no model schema |
| Tenant setting: *...with DAX and mashup expressions* | §2 returns no measures or M queries |
| Capacity Metrics app + read access | §4b skipped — no throttling evidence |
| Workspace access to Warehouses / SQL endpoints | §6 partial or skipped |
| Approved data access | Part Two not runnable |
| Custom Environment with `great_expectations` | §11 falls back to a native engine |
| A default Lakehouse attached | Results not persisted |

Every section degrades gracefully. §14 prints an explicit **coverage gaps** list — read it before presenting anything, so you never claim completeness you did not achieve.

## Two warnings

1. **This consumes capacity units.** A full-tenant scan on the shared production capacity is exactly the noisy-neighbour behaviour worth avoiding. Run it on a dev capacity.
2. **`COUNT_ROWS` and `SAMPLE_FRACTION`** control cost. Leave `COUNT_ROWS = False` on a first pass.

Read-only throughout. Nothing is created, modified or deleted outside the `estate_*` output tables.


In [ ]:
SCOPE                 = 'Tenant'   # 'Tenant' uses admin APIs, 'User' only what you can see
RUN_SCANNER           = True       # deep metadata: relationships, RLS, lineage - needs Fabric Admin
PROFILE_DELTA         = True       # Delta internals: rebuild vs MERGE, small files
INCLUDE_DEFINITIONS   = True       # pipeline / notebook definitions - slowest section
RUN_SQL_ENDPOINT      = True       # views vs persisted tables, view chain depth
COUNT_ROWS            = False      # expensive: full scan per table
MAX_TABLES_PROFILED   = 1000
WORKSPACE_NAME_FILTER = ''         # substring filter, blank = all

SAVE_TO_LAKEHOUSE     = True
TABLE_PREFIX          = 'estate_'

# Only needed if you are NOT a Fabric Admin and want to run as a service principal
KEYVAULT_URL          = ''
KV_SECRET_CLIENT_ID   = ''
KV_SECRET_CLIENT_KEY  = ''
SP_TENANT_ID          = ''


In [ ]:
import json, time, datetime as dt
import requests
import pandas as pd
import notebookutils

FABRIC_API = 'https://api.fabric.microsoft.com/v1'
PBI_API    = 'https://api.powerbi.com/v1.0/myorg'
SCAN_START = dt.datetime.utcnow().isoformat()


def _sp_token():
    cid = notebookutils.credentials.getSecret(KEYVAULT_URL, KV_SECRET_CLIENT_ID)
    key = notebookutils.credentials.getSecret(KEYVAULT_URL, KV_SECRET_CLIENT_KEY)
    url = 'https://login.microsoftonline.com/' + SP_TENANT_ID + '/oauth2/v2.0/token'
    body = {
        'grant_type': 'client_credentials',
        'client_id': cid,
        'client_secret': key,
        'scope': 'https://analysis.windows.net/powerbi/api/.default',
    }
    r = requests.post(url, data=body, timeout=60)
    r.raise_for_status()
    return r.json()['access_token']


if KEYVAULT_URL and SP_TENANT_ID:
    TOKEN = _sp_token()
    print('Authenticated as service principal')
else:
    TOKEN = notebookutils.credentials.getToken('pbi')
    print('Authenticated as the notebook executor')

# Fabric and Power BI APIs both accept the Power BI audience token
HEADERS = {'Authorization': 'Bearer ' + TOKEN, 'Content-Type': 'application/json'}


def api(url, method='GET', body=None, max_retry=5):
    for attempt in range(1, max_retry + 1):
        try:
            if method == 'POST':
                r = requests.post(url, headers=HEADERS, json=body, timeout=180)
            else:
                r = requests.get(url, headers=HEADERS, timeout=180)
        except requests.RequestException:
            time.sleep(min(60, 2 ** attempt))
            continue

        if r.status_code == 429:
            wait = int(r.headers.get('Retry-After', 30))
            print('  throttled, waiting ' + str(wait) + 's')
            time.sleep(wait)
            continue
        if r.status_code in (401, 403, 404):
            return None
        if r.status_code >= 500:
            time.sleep(min(60, 2 ** attempt))
            continue
        if not r.ok:
            return None
        return r.json() if r.content else {}
    return None


def api_paged(url):
    out, nxt = [], url
    while nxt:
        page = api(nxt)
        if not page:
            break
        out.extend(page.get('value', []))
        nxt = page.get('continuationUri')
    return out


COLLECTED = {}


def collect(name, records):
    df = pd.DataFrame(records)
    if not df.empty:
        df['Provenance'] = 'Tool-scanned'
    COLLECTED[name] = df
    print('  ' + name + ': ' + str(len(df)) + ' rows')
    return df


## 1 · Tenant inventory

Capacities, workspaces, items, OneLake tables and shortcuts. Everything here works without admin rights, though `SCOPE = 'User'` limits it to workspaces you can see.


In [ ]:
print('Capacities and workspaces')

caps = api_paged(FABRIC_API + '/admin/capacities') if SCOPE == 'Tenant' else []
if not caps:
    caps = api_paged(FABRIC_API + '/capacities')

cap_df = collect('capacities', [{
    'CapacityId': c.get('id'), 'DisplayName': c.get('displayName'),
    'Sku': c.get('sku'), 'Region': c.get('region'), 'State': c.get('state'),
} for c in caps])

cap_lookup = {c.get('id'): c for c in caps}

ws = api_paged(FABRIC_API + '/admin/workspaces') if SCOPE == 'Tenant' else []
if not ws:
    print('  admin scope unavailable, falling back to user scope')
    ws = api_paged(FABRIC_API + '/workspaces')

ws_df = collect('workspaces', [{
    'WorkspaceId': w.get('id'), 'DisplayName': w.get('displayName'),
    'Type': w.get('type'), 'State': w.get('state'),
    'CapacityId': w.get('capacityId'),
    'CapacityName': cap_lookup.get(w.get('capacityId'), {}).get('displayName'),
    'CapacitySku': cap_lookup.get(w.get('capacityId'), {}).get('sku'),
    'Description': w.get('description'),
} for w in ws])

active = [w for w in ws
          if w.get('state') != 'Deleted'
          and w.get('type') != 'PersonalWorkspace'
          and (not WORKSPACE_NAME_FILTER
               or WORKSPACE_NAME_FILTER.lower() in (w.get('displayName') or '').lower())]
print('  ' + str(len(active)) + ' workspaces to enumerate')

print('Items')
all_items, item_rows = [], []
for w in active:
    for it in api_paged(FABRIC_API + '/workspaces/' + w['id'] + '/items'):
        all_items.append({'ws': w, 'item': it})
        item_rows.append({
            'WorkspaceId': w['id'], 'WorkspaceName': w.get('displayName'),
            'ItemId': it.get('id'), 'ItemName': it.get('displayName'),
            'ItemType': it.get('type'), 'Description': it.get('description'),
        })

items_df = collect('items', item_rows)
if not items_df.empty:
    display(items_df.groupby('ItemType').size().sort_values(ascending=False).to_frame('Count'))


In [ ]:
print('OneLake tables and shortcuts')

lakehouses = [a for a in all_items if a['item'].get('type') == 'Lakehouse']

table_rows = []
for lh in lakehouses:
    wid, lid = lh['ws']['id'], lh['item']['id']
    for t in api_paged(FABRIC_API + '/workspaces/' + wid + '/lakehouses/' + lid + '/tables'):
        table_rows.append({
            'WorkspaceId': wid, 'WorkspaceName': lh['ws'].get('displayName'),
            'LakehouseId': lid, 'LakehouseName': lh['item'].get('displayName'),
            'TableName': t.get('name'), 'TableType': t.get('type'),
            'Format': t.get('format'), 'Location': t.get('location'),
        })
tables_df = collect('lakehouse_tables', table_rows)


def shortcut_detail(tgt):
    kind = tgt.get('type')
    if not kind:
        return ''
    d = tgt.get(kind[0].lower() + kind[1:], {}) or {}
    if kind == 'OneLake':
        return 'ws=' + str(d.get('workspaceId')) + ';item=' + str(d.get('itemId')) + ';path=' + str(d.get('path'))
    return str(d.get('location', '')) + str(d.get('subpath', '')) + str(d.get('bucket', ''))


sc_rows = []
hosts = [a for a in all_items if a['item'].get('type') in ('Lakehouse', 'Warehouse', 'KQLDatabase')]
for h in hosts:
    wid, iid = h['ws']['id'], h['item']['id']
    for s in api_paged(FABRIC_API + '/workspaces/' + wid + '/items/' + iid + '/shortcuts'):
        tgt = s.get('target', {})
        sc_rows.append({
            'WorkspaceName': h['ws'].get('displayName'),
            'HostItem': h['item'].get('displayName'),
            'HostItemType': h['item'].get('type'),
            'ShortcutName': s.get('name'), 'ShortcutPath': s.get('path'),
            'TargetType': tgt.get('type'), 'TargetDetail': shortcut_detail(tgt),
            'IsExternal': tgt.get('type') != 'OneLake',
            'IsTransform': bool(s.get('isShortcutTransform')),
        })
sc_df = collect('shortcuts', sc_rows)

if not sc_df.empty:
    display(sc_df.groupby(['TargetType', 'IsExternal']).size().to_frame('Count'))


## 2 · Deep metadata — Scanner API

Semantic model schema, **relationships**, RLS filter expressions, M queries, lineage and data sources.

Requires Fabric Administrator. If the call is rejected the cells print a warning and continue — you keep sections 1, 3 and 4.


In [ ]:
scan_results = []

if RUN_SCANNER:
    ids = [w['id'] for w in active]
    batches = [ids[i:i + 100] for i in range(0, len(ids), 100)]
    print('Scanner API: ' + str(len(batches)) + ' batch(es)')

    scan_url = (PBI_API + '/admin/workspaces/getInfo'
                + '?lineage=True&datasourceDetails=True'
                + '&datasetSchema=True&datasetExpressions=True&getArtifactUsers=True')

    for n, batch in enumerate(batches, start=1):
        started = api(scan_url, method='POST', body={'workspaces': batch})
        if not started:
            print('  REJECTED - not a Fabric Admin, or metadata scanning tenant settings are off')
            break

        sid, status, waited = started['id'], started.get('status'), 0
        while status not in ('Succeeded', 'Failed') and waited < 900:
            time.sleep(5)
            waited += 5
            st = api(PBI_API + '/admin/workspaces/scanStatus/' + sid)
            if not st:
                break
            status = st.get('status')

        if status != 'Succeeded':
            print('  batch ' + str(n) + ' ended as ' + str(status))
            continue

        res = api(PBI_API + '/admin/workspaces/scanResult/' + sid)
        if res:
            scan_results.append(res)
            print('  batch ' + str(n) + '/' + str(len(batches)) + ' ok')
else:
    print('Scanner skipped by parameter')


In [ ]:
models, cols, measures, rels, rls, exprs = [], [], [], [], [], []
lineage, dsrc, reports, dflows = [], [], [], []

for res in scan_results:
    bad = res.get('misconfiguredDatasourceInstances', []) or []
    for d in (res.get('datasourceInstances', []) or []) + bad:
        cd = d.get('connectionDetails', {}) or {}
        dsrc.append({
            'DatasourceId': d.get('datasourceId'), 'DatasourceType': d.get('datasourceType'),
            'Server': cd.get('server'), 'Database': cd.get('database'),
            'Path': cd.get('path'), 'Url': cd.get('url'),
            'GatewayId': d.get('gatewayId'), 'Misconfigured': d in bad,
        })

    for w in res.get('workspaces', []):
        wn = w.get('name')

        for d in w.get('datasets', []):
            models.append({
                'WorkspaceName': wn, 'DatasetId': d.get('id'), 'DatasetName': d.get('name'),
                'ConfiguredBy': d.get('configuredBy'), 'StorageMode': d.get('targetStorageMode'),
                'ContentProvider': d.get('ContentProviderType'),
                'Endorsement': (d.get('endorsementDetails') or {}).get('endorsement'),
                'SensitivityLabel': (d.get('sensitivityLabel') or {}).get('labelId'),
                'TableCount': len(d.get('tables', []) or []),
                'RelationshipCount': len(d.get('relationships', []) or []),
                'RoleCount': len(d.get('roles', []) or []),
                'SchemaStale': d.get('schemaMayNotBeUpToDate'),
            })

            for r in (d.get('relationships') or []):
                rels.append({
                    'WorkspaceName': wn, 'DatasetName': d.get('name'),
                    'RelationshipName': r.get('name'),
                    'FromTable': r.get('fromTable'), 'FromColumn': r.get('fromColumn'),
                    'FromCardinality': r.get('fromCardinality'),
                    'ToTable': r.get('toTable'), 'ToColumn': r.get('toColumn'),
                    'ToCardinality': r.get('toCardinality'),
                    'CrossFilter': r.get('crossFilteringBehavior'),
                    'IsActive': r.get('isActive'),
                })

            for role in (d.get('roles') or []):
                members = '; '.join([m.get('memberName', '') for m in (role.get('members') or [])])
                for tp in (role.get('tablePermissions') or [{}]):
                    rls.append({
                        'WorkspaceName': wn, 'DatasetName': d.get('name'),
                        'RoleName': role.get('name'),
                        'ModelPermission': role.get('modelPermission'),
                        'Members': members, 'MemberCount': len(role.get('members') or []),
                        'FilteredTable': tp.get('name'),
                        'FilterExpression': tp.get('filterExpression'),
                    })

            for e in (d.get('expressions') or []):
                exprs.append({'WorkspaceName': wn, 'DatasetName': d.get('name'),
                              'ExpressionName': e.get('name'), 'Expression': e.get('expression')})

            for t in (d.get('tables') or []):
                for c in (t.get('columns') or []):
                    cols.append({'WorkspaceName': wn, 'DatasetName': d.get('name'),
                                 'TableName': t.get('name'), 'ColumnName': c.get('name'),
                                 'DataType': c.get('dataType'), 'IsHidden': c.get('isHidden')})
                for m in (t.get('measures') or []):
                    measures.append({'WorkspaceName': wn, 'DatasetName': d.get('name'),
                                     'TableName': t.get('name'), 'MeasureName': m.get('name'),
                                     'Expression': m.get('expression'), 'IsHidden': m.get('isHidden')})

            for key, kind in (('upstreamDataflows', 'Dataflow'),
                              ('upstreamDatasets', 'SemanticModel'),
                              ('upstreamDatamarts', 'Datamart')):
                for u in (d.get(key) or []):
                    lineage.append({
                        'WorkspaceName': wn, 'Downstream': d.get('name'), 'DownstreamId': d.get('id'),
                        'DownstreamType': 'SemanticModel', 'UpstreamType': kind,
                        'UpstreamId': u.get('targetDataflowId') or u.get('targetDatasetId') or u.get('targetDatamartId'),
                    })
            for u in (d.get('datasourceUsages') or []):
                lineage.append({'WorkspaceName': wn, 'Downstream': d.get('name'), 'DownstreamId': d.get('id'),
                                'DownstreamType': 'SemanticModel', 'UpstreamType': 'Datasource',
                                'UpstreamId': u.get('datasourceInstanceId')})

        for r in w.get('reports', []):
            reports.append({
                'WorkspaceName': wn, 'ReportId': r.get('id'), 'ReportName': r.get('name'),
                'ReportType': r.get('reportType'), 'DatasetId': r.get('datasetId'),
                'DatasetWorkspaceId': r.get('datasetWorkspaceId'),
                'IsCrossWorkspace': bool(r.get('datasetWorkspaceId')),
                'ModifiedBy': r.get('modifiedBy'), 'ModifiedUtc': r.get('modifiedDateTime'),
            })
            lineage.append({'WorkspaceName': wn, 'Downstream': r.get('name'), 'DownstreamId': r.get('id'),
                            'DownstreamType': 'Report', 'UpstreamType': 'SemanticModel',
                            'UpstreamId': r.get('datasetId')})

        for df in w.get('dataflows', []):
            dflows.append({'WorkspaceName': wn, 'DataflowId': df.get('objectId'),
                           'DataflowName': df.get('name'), 'ConfiguredBy': df.get('configuredBy'),
                           'ModifiedUtc': df.get('modifiedDateTime')})

collect('semantic_models', models)
collect('model_columns', cols)
collect('model_measures', measures)
collect('model_relationships', rels)
collect('rls_roles', rls)
collect('model_expressions', exprs)
collect('lineage', lineage)
collect('datasources', dsrc)
collect('reports', reports)
collect('dataflows', dflows)


## 3 · Delta profiling — the part only a notebook can do

For every OneLake table: real schema, file count, size, partitioning, and the **maintenance pattern** read from the Delta transaction log.

`MaintenancePattern` is the column to read first:

| Value | Means |
|---|---|
| `INCREMENTAL (MERGE)` | Healthy — only changed rows are written |
| `FULL REBUILD (Overwrite)` | The whole table is rewritten every run. On a large Gold table that is a capacity problem hiding in plain sight. |
| `APPEND` | Expected for Bronze. Suspicious for a Gold dimension. |

Combined with the Gold-as-views finding, this tells you whether Gold is *persisted but rebuilt* — which costs nearly as much as views and looks fine on an architecture diagram.


In [ ]:
delta_rows, schema_rows, hist_rows = [], [], []

if PROFILE_DELTA and not tables_df.empty:
    recs = tables_df.to_dict('records')[:MAX_TABLES_PROFILED]
    print('Profiling ' + str(len(recs)) + ' Delta tables')

    for n, t in enumerate(recs, start=1):
        path = t.get('Location')
        if not path:
            path = ('abfss://' + t['WorkspaceId'] + '@onelake.dfs.fabric.microsoft.com/'
                    + t['LakehouseId'] + '/Tables/' + t['TableName'])
        if n % 50 == 0:
            print('  ' + str(n) + '/' + str(len(recs)))

        base = {'WorkspaceName': t['WorkspaceName'], 'LakehouseName': t['LakehouseName'],
                'TableName': t['TableName'], 'Path': path}
        try:
            detail = spark.sql('DESCRIBE DETAIL delta.`' + path + '`').collect()[0].asDict()
            num_files = detail.get('numFiles') or 0
            size_mb = round((detail.get('sizeInBytes') or 0) / 1048576.0, 2)
            avg_mb = round(size_mb / num_files, 2) if num_files else 0.0
            parts = detail.get('partitionColumns') or []

            hist = spark.sql('DESCRIBE HISTORY delta.`' + path + '` LIMIT 25').collect()
            ops = [h['operation'] for h in hist]
            last = hist[0].asDict() if hist else {}
            mode = (last.get('operationParameters') or {}).get('mode', '')

            if 'MERGE' in ops:
                pattern = 'INCREMENTAL (MERGE)'
            elif mode == 'Overwrite' or 'CREATE OR REPLACE TABLE AS SELECT' in ops:
                pattern = 'FULL REBUILD (Overwrite)'
            elif mode == 'Append':
                pattern = 'APPEND'
            elif ops:
                pattern = ops[0]
            else:
                pattern = 'UNKNOWN'

            row_count = spark.read.format('delta').load(path).count() if COUNT_ROWS else None

            delta_rows.append(dict(base, **{
                'NumFiles': num_files, 'SizeMB': size_mb, 'AvgFileMB': avg_mb,
                'PartitionColumns': ', '.join(parts), 'IsPartitioned': len(parts) > 0,
                'RowCount': row_count,
                'LastOperation': last.get('operation'), 'LastOperationMode': mode,
                'LastWriteUtc': str(last.get('timestamp')), 'LastWriteBy': last.get('userName'),
                'MaintenancePattern': pattern,
                'RecentOps': ', '.join(sorted(set(ops))),
                'HasOptimize': 'OPTIMIZE' in ops,
                'SmallFileRisk': bool(num_files > 100 and avg_mb < 16),
                'ProfileError': None,
            }))

            for f in spark.read.format('delta').load(path).schema.fields:
                schema_rows.append(dict(base, **{
                    'ColumnName': f.name, 'DataType': f.dataType.simpleString(), 'Nullable': f.nullable,
                }))

            for h in hist[:10]:
                hd = h.asDict()
                hist_rows.append(dict(base, **{
                    'Version': hd.get('version'), 'TimestampUtc': str(hd.get('timestamp')),
                    'Operation': hd.get('operation'), 'UserName': hd.get('userName'),
                    'Mode': (hd.get('operationParameters') or {}).get('mode'),
                    'Metrics': json.dumps(hd.get('operationMetrics') or {}),
                }))

        except Exception as ex:
            delta_rows.append(dict(base, **{'ProfileError': str(ex)[:400]}))

delta_df = collect('delta_profile', delta_rows)
collect('delta_schema', schema_rows)
collect('delta_history', hist_rows)

if not delta_df.empty and 'MaintenancePattern' in delta_df.columns:
    display(delta_df.groupby('MaintenancePattern').size().sort_values(ascending=False).to_frame('Tables'))
    big = delta_df[delta_df['MaintenancePattern'] == 'FULL REBUILD (Overwrite)']
    if not big.empty:
        display(big.nlargest(20, 'SizeMB')[['WorkspaceName', 'LakehouseName', 'TableName', 'SizeMB', 'NumFiles', 'LastWriteUtc']])


## 4 · Governance topology

Domains, Git connections, deployment pipelines, workspace role assignments, scheduled jobs, external data shares, connections and tenant settings.

Tenant settings matter more than they look: if *Enhance admin APIs responses with detailed metadata* is off, that explains any gaps in section 2 — and it is a change-control item at a regulated organisation, not a toggle someone flips in a meeting.


In [ ]:
print('Governance topology')

dom_rows = []
for d in api_paged(FABRIC_API + '/admin/domains'):
    for w in (api_paged(FABRIC_API + '/admin/domains/' + d['id'] + '/workspaces') or [{}]):
        dom_rows.append({'DomainId': d.get('id'), 'DomainName': d.get('displayName'),
                         'ParentDomainId': d.get('parentDomainId'),
                         'WorkspaceId': w.get('id'), 'WorkspaceName': w.get('displayName')})
collect('domains', dom_rows)

git_rows = []
for w in active:
    g = api(FABRIC_API + '/workspaces/' + w['id'] + '/git/connection')
    if not g:
        continue
    p = g.get('gitProviderDetails') or {}
    git_rows.append({'WorkspaceName': w.get('displayName'), 'GitProvider': p.get('gitProviderType'),
                     'Organization': p.get('organizationName'), 'Project': p.get('projectName'),
                     'Repository': p.get('repositoryName'), 'Branch': p.get('branchName'),
                     'ConnectionState': g.get('gitConnectionState')})
git_df = collect('git_connections', git_rows)

dep_pipe_rows = []
for dp in api_paged(FABRIC_API + '/deploymentPipelines'):
    for st in api_paged(FABRIC_API + '/deploymentPipelines/' + dp['id'] + '/stages'):
        dep_pipe_rows.append({'PipelineName': dp.get('displayName'), 'StageOrder': st.get('order'),
                              'StageName': st.get('displayName'), 'WorkspaceName': st.get('workspaceName')})
collect('deployment_pipelines', dep_pipe_rows)

role_rows = []
for w in active:
    for ra in api_paged(FABRIC_API + '/workspaces/' + w['id'] + '/roleAssignments'):
        pr = ra.get('principal', {}) or {}
        role_rows.append({'WorkspaceName': w.get('displayName'), 'PrincipalName': pr.get('displayName'),
                          'PrincipalType': pr.get('type'), 'Role': ra.get('role')})
collect('workspace_roles', role_rows)

job_map = {'DataPipeline': 'Pipeline', 'Notebook': 'RunNotebook',
           'SparkJobDefinition': 'sparkjob', 'CopyJob': 'CopyJob'}
sched_rows = []
for a in all_items:
    jt = job_map.get(a['item'].get('type'))
    if not jt:
        continue
    url = (FABRIC_API + '/workspaces/' + a['ws']['id'] + '/items/'
           + a['item']['id'] + '/jobs/' + jt + '/schedules')
    for s in api_paged(url):
        cfg = s.get('configuration', {}) or {}
        sched_rows.append({'WorkspaceName': a['ws'].get('displayName'),
                           'ItemName': a['item'].get('displayName'),
                           'ItemType': a['item'].get('type'), 'Enabled': s.get('enabled'),
                           'RecurrenceType': cfg.get('type'), 'Interval': cfg.get('interval'),
                           'Times': ', '.join(cfg.get('times') or []),
                           'TimeZone': cfg.get('localTimeZoneId')})
sched_df = collect('job_schedules', sched_rows)

collect('external_data_shares', [{
    'ShareId': e.get('id'), 'ItemId': e.get('itemId'), 'WorkspaceId': e.get('workspaceId'),
    'Paths': ', '.join(e.get('paths') or []), 'Status': e.get('status'),
    'RecipientTenant': (e.get('recipient') or {}).get('tenantId'),
} for e in api_paged(FABRIC_API + '/admin/items/externalDataShares')])

collect('connections', [{
    'ConnectionId': c.get('id'), 'DisplayName': c.get('displayName'),
    'ConnectivityType': c.get('connectivityType'), 'GatewayId': c.get('gatewayId'),
    'PrivacyLevel': c.get('privacyLevel'),
    'Path': (c.get('connectionDetails') or {}).get('path'),
    'Type': (c.get('connectionDetails') or {}).get('type'),
    'CredentialType': (c.get('credentialDetails') or {}).get('credentialType'),
} for c in api_paged(FABRIC_API + '/connections')])

ts = api(FABRIC_API + '/admin/tenantsettings') if SCOPE == 'Tenant' else None
collect('tenant_settings', [{
    'SettingName': s.get('settingName'), 'Title': s.get('title'), 'Enabled': s.get('enabled'),
    'Group': s.get('tenantSettingGroup'), 'DelegateToWorkspace': s.get('delegateToWorkspace'),
    'EnabledSecurityGroups': '; '.join([g.get('name', '') for g in (s.get('enabledSecurityGroups') or [])]),
} for s in ((ts or {}).get('tenantSettings') or [])])


## 4b · Capacity metrics and throttling evidence

Section 1 tells you *which workspace sits on which SKU* — the isolation question. This section tells you **what actually happened to that capacity**: utilisation, throttling, smoothing and the top consuming operations.

That evidence lives in the **Fabric Capacity Metrics** app's semantic model, not in any REST API. This cell locates it and queries it with DAX via `sempy`.

**Requires:** the Capacity Metrics app installed, and the runner having read access to it (normally Capacity Admin).

The app's internal model changes between versions, so the cell first prints the tables and measures it actually found, then attempts a set of candidate queries. Anything that fails is reported rather than raised — you get whatever the installed version exposes.


In [ ]:
cap_model, cap_ws, cap_tables, cap_measures = None, None, [], []
cap_metric_rows, cap_query_log = [], []

try:
    import sempy.fabric as fabric

    ds_list = fabric.list_datasets(mode='rest')
    hits = ds_list[ds_list['Dataset Name'].str.contains('Capacity Metrics', case=False, na=False)]
    if len(hits) == 0:
        for w in active:
            try:
                d = fabric.list_datasets(workspace=w['id'], mode='rest')
                h = d[d['Dataset Name'].str.contains('Capacity Metrics', case=False, na=False)]
                if len(h):
                    cap_model, cap_ws = h.iloc[0]['Dataset Name'], w['id']
                    break
            except Exception:
                continue
    else:
        cap_model = hits.iloc[0]['Dataset Name']
        cap_ws = hits.iloc[0].get('Workspace Id')

    if not cap_model:
        print('Fabric Capacity Metrics semantic model not found or not accessible')
    else:
        print('Found model: ' + str(cap_model))
        try:
            t = fabric.list_tables(cap_model, workspace=cap_ws)
            cap_tables = t['Name'].tolist()
            print('  tables: ' + ', '.join(cap_tables[:25]))
        except Exception as ex:
            print('  could not list tables: ' + str(ex)[:150])
        try:
            m = fabric.list_measures(cap_model, workspace=cap_ws)
            cap_measures = m['Measure Name'].tolist()
            print('  measures: ' + str(len(cap_measures)))
        except Exception as ex:
            print('  could not list measures: ' + str(ex)[:150])

        def try_dax(label, dax):
            try:
                df = fabric.evaluate_dax(cap_model, dax, workspace=cap_ws)
                cap_query_log.append({'Query': label, 'Status': 'ok', 'Rows': len(df)})
                print('  ok  ' + label + '  (' + str(len(df)) + ' rows)')
                return df
            except Exception as ex:
                cap_query_log.append({'Query': label, 'Status': 'failed', 'Error': str(ex)[:250]})
                print('  --  ' + label + '  ' + str(ex)[:110])
                return None

        # The app's model differs between versions; each candidate is attempted independently.
        caps_df = try_dax('capacities', 'EVALUATE SELECTCOLUMNS( \'Capacities\', "CapacityId", \'Capacities\'[capacityId], "Name", \'Capacities\'[Capacity Name], "Sku", \'Capacities\'[SKU], "Region", \'Capacities\'[Region] )')
        if caps_df is not None:
            for _, r in caps_df.iterrows():
                cap_metric_rows.append({'Metric': 'CapacityRegistered', 'CapacityId': str(r.get('[CapacityId]')),
                                        'CapacityName': str(r.get('[Name]')), 'Value': str(r.get('[Sku]')),
                                        'Detail': str(r.get('[Region]'))})

        util = try_dax('utilisation_by_day',
                       'EVALUATE SUMMARIZECOLUMNS( \'TimePoints\'[Date], \'Capacities\'[Capacity Name], '
                       '"CU_pct", [Dynamic CU Utilization %], "Throttling_pct", [Throttling %] )')
        if util is None:
            util = try_dax('utilisation_by_day_alt',
                           'EVALUATE SUMMARIZECOLUMNS( \'MetricsByItemandDay\'[Date], '
                           '"CU_seconds", SUM( \'MetricsByItemandDay\'[CU (s)] ) )')
        if util is not None:
            for _, r in util.iterrows():
                d = {k: str(v) for k, v in r.items()}
                cap_metric_rows.append({'Metric': 'DailyUtilisation', 'CapacityId': '',
                                        'CapacityName': d.get('Capacities[Capacity Name]', ''),
                                        'Value': d.get('[CU_pct]', d.get('[CU_seconds]', '')),
                                        'Detail': json.dumps(d)[:900]})

        top = try_dax('top_consuming_items',
                      'EVALUATE TOPN( 50, SUMMARIZECOLUMNS( \'Items\'[WorkspaceName], \'Items\'[ItemName], '
                      '\'Items\'[ItemKind], "CU_seconds", SUM( \'MetricsByItemandDay\'[CU (s)] ) ), '
                      '[CU_seconds], DESC )')
        if top is not None:
            for _, r in top.iterrows():
                d = {k: str(v) for k, v in r.items()}
                cap_metric_rows.append({'Metric': 'TopConsumingItem', 'CapacityId': '',
                                        'CapacityName': d.get('Items[WorkspaceName]', ''),
                                        'Value': d.get('[CU_seconds]', ''),
                                        'Detail': json.dumps(d)[:900]})

        thr = try_dax('throttling_events',
                      'EVALUATE FILTER( SUMMARIZECOLUMNS( \'TimePoints\'[TimePoint], \'Capacities\'[Capacity Name], '
                      '"Throttling", [Throttling %] ), [Throttling] > 0 )')
        if thr is not None:
            for _, r in thr.iterrows():
                d = {k: str(v) for k, v in r.items()}
                cap_metric_rows.append({'Metric': 'ThrottlingEvent', 'CapacityId': '',
                                        'CapacityName': d.get('Capacities[Capacity Name]', ''),
                                        'Value': d.get('[Throttling]', ''),
                                        'Detail': json.dumps(d)[:900]})

except ImportError:
    print('sempy is not available in this runtime - capacity metrics skipped')
except Exception as ex:
    print('Capacity metrics section failed: ' + str(ex)[:300])

cap_metrics_df = collect('capacity_metrics', cap_metric_rows)
collect('capacity_metrics_querylog', cap_query_log)
collect('capacity_metrics_modelinfo',
        [{'Model': str(cap_model), 'Tables': '; '.join(cap_tables),
          'MeasureCount': len(cap_measures), 'Measures': '; '.join(cap_measures[:200])}] if cap_model else [])

throttle_events = 0
if not cap_metrics_df.empty:
    throttle_events = int((cap_metrics_df['Metric'] == 'ThrottlingEvent').sum())
    display(cap_metrics_df.groupby('Metric').size().to_frame('Rows'))
    top_rows = cap_metrics_df[cap_metrics_df['Metric'] == 'TopConsumingItem']
    if not top_rows.empty:
        display(top_rows.head(20)[['CapacityName', 'Value', 'Detail']])
print('Throttling events captured: ' + str(throttle_events))


## 5 · Item definitions — the real data flows

Downloads pipeline JSON, notebook source and semantic model TMDL, then parses them.

This is how you reconstruct the flows that **actually exist** rather than the ones people describe in the room. Pipeline activities are walked recursively through `ForEach` / `If` / `Switch` containers, extracting every source, sink and dependency.

Set `INCLUDE_DEFINITIONS = False` in the parameters cell to skip — it is the slowest section.


In [ ]:
import base64, re

DEFINITION_TYPES = ('DataPipeline', 'Notebook', 'SemanticModel', 'Dataflow',
                    'CopyJob', 'Eventstream', 'SparkJobDefinition')
MAX_TEXT_PER_PART = 400000

# decoded source retained for the lineage parser in section 6b
DEF_TEXT = {}


def get_definition(ws_id, item_id):
    url = FABRIC_API + '/workspaces/' + ws_id + '/items/' + item_id + '/getDefinition'
    try:
        r = requests.post(url, headers=HEADERS, timeout=300)
    except requests.RequestException:
        return None
    if r.status_code == 202:
        op = r.headers.get('Location')
        for _ in range(80):
            time.sleep(3)
            state = api(op)
            if not state:
                return None
            if state.get('status') == 'Succeeded':
                return api(op + '/result')
            if state.get('status') == 'Failed':
                return None
        return None
    if r.ok and r.content:
        return r.json()
    return None


def walk_activities(acts, parent=''):
    rows = []
    for a in (acts or []):
        name = a.get('name', '')
        path = (parent + ' > ' + name) if parent else name
        tp = a.get('typeProperties') or {}
        src, snk = tp.get('source') or {}, tp.get('sink') or {}

        invokes = ''
        if tp.get('notebookId'):
            invokes = 'notebook:' + str(tp.get('notebookId'))
        elif (tp.get('pipeline') or {}).get('referenceName'):
            invokes = 'pipeline:' + str(tp['pipeline']['referenceName'])
        elif tp.get('dataflowId'):
            invokes = 'dataflow:' + str(tp.get('dataflowId'))

        rows.append({
            'ActivityPath': path, 'ActivityName': name, 'ActivityType': a.get('type'),
            'DependsOn': '; '.join([str(d.get('activity')) + str(d.get('dependencyConditions'))
                                    for d in (a.get('dependsOn') or [])]),
            'SourceType': src.get('type'), 'SinkType': snk.get('type'),
            'SourceDetail': json.dumps(src)[:4000], 'SinkDetail': json.dumps(snk)[:4000],
            'Invokes': invokes,
            'InputRefs': '; '.join([str(i.get('referenceName')) for i in (a.get('inputs') or [])]),
            'OutputRefs': '; '.join([str(o.get('referenceName')) for o in (a.get('outputs') or [])]),
        })

        for child in (tp.get('activities'), tp.get('ifTrueActivities'),
                      tp.get('ifFalseActivities'), tp.get('defaultActivities')):
            rows.extend(walk_activities(child, path))
        for case in (tp.get('cases') or []):
            rows.extend(walk_activities(case.get('activities'), path + '[' + str(case.get('value')) + ']'))
    return rows


act_rows, nb_rows, def_rows = [], [], []

if INCLUDE_DEFINITIONS:
    targets = [a for a in all_items if a['item'].get('type') in DEFINITION_TYPES]
    print('Downloading ' + str(len(targets)) + ' item definitions')

    for n, t in enumerate(targets, start=1):
        if n % 25 == 0:
            print('  ' + str(n) + '/' + str(len(targets)))
        payload = get_definition(t['ws']['id'], t['item']['id'])
        if not payload:
            continue

        itype = t['item'].get('type')
        iname = t['item'].get('displayName')
        wname = t['ws'].get('displayName')

        for part in ((payload.get('definition') or {}).get('parts') or []):
            try:
                raw = base64.b64decode(part.get('payload') or '')
            except Exception:
                continue
            def_rows.append({'WorkspaceName': wname, 'ItemName': iname, 'ItemType': itype,
                             'PartPath': part.get('path'), 'Bytes': len(raw)})
            text = raw.decode('utf-8', errors='ignore')

            if itype in ('Notebook', 'DataPipeline', 'SparkJobDefinition') and text:
                DEF_TEXT[(wname, iname, itype, str(part.get('path')))] = text[:MAX_TEXT_PER_PART]

            if itype == 'DataPipeline' and 'pipeline-content' in str(part.get('path')):
                try:
                    pj = json.loads(text)
                    acts = (pj.get('properties') or {}).get('activities') or pj.get('activities')
                    for a in walk_activities(acts):
                        act_rows.append(dict({'WorkspaceName': wname, 'PipelineName': iname}, **a))
                except Exception:
                    pass

            if itype == 'Notebook' and text:
                lh = sorted(set(re.findall(r'"default_lakehouse_name"\s*:\s*"([^"]+)"', text)))
                abfss = sorted(set(re.findall(r'abfss://[^\s"\'\)]+', text)))[:30]
                tbls = sorted(set(re.findall(
                    r'(?i)(?:saveAsTable|spark\.read\.table|\bFROM\b|\bINTO\b)\s*\(?\s*["\']([A-Za-z0-9_\.]+)["\']',
                    text)))[:50]
                nb_rows.append({'WorkspaceName': wname, 'NotebookName': iname,
                                'DefaultLakehouse': '; '.join(lh),
                                'AbfssPaths': '; '.join(abfss),
                                'TableReferences': '; '.join(tbls),
                                'TableRefCount': len(tbls)})
else:
    print('INCLUDE_DEFINITIONS is False - skipping')

collect('item_definitions', def_rows)
act_df = collect('pipeline_activities', act_rows)
collect('notebook_references', nb_rows)
print('  source text retained for lineage parsing: ' + str(len(DEF_TEXT)) + ' parts')

if not act_df.empty:
    display(act_df.groupby('ActivityType').size().sort_values(ascending=False).to_frame('Count'))


## 6 · SQL endpoint objects — views vs persisted tables

**The single most important section for an architecture assessment, and the one everything else is blind to.**

Views have no Delta transaction log and are not returned by the Lakehouse tables API. A Warehouse full of chained views over Silver produces *nothing* from sections 1 and 3 — and you would read that silence as "no Gold objects" rather than "every Gold object is a view."

This section connects to each Warehouse and Lakehouse SQL analytics endpoint and reads `sys.objects` and `sys.sql_expression_dependencies` directly, then computes **view chain depth** in Python.

| Depth | Meaning |
|---|---|
| 0 | Persisted table |
| 1 | View over tables |
| 2+ | **View over view** — every read recomputes the whole chain |

Depth ≥ 2 on a sponsor-facing object is the Roche incident, expressed as a number.

Connection uses your Entra identity via ODBC token auth. If `pyodbc` or the driver is unavailable the cell degrades and tells you what it could not reach.


In [ ]:
import struct

SQL_OBJECTS_Q = """
SELECT s.name AS schema_name, o.name AS object_name,
       CASE WHEN o.type = 'V' THEN 'VIEW' ELSE 'TABLE' END AS object_kind,
       o.create_date, o.modify_date
FROM sys.objects o
JOIN sys.schemas s ON s.schema_id = o.schema_id
WHERE o.type IN ('U','V') AND o.is_ms_shipped = 0
"""

SQL_DEPS_Q = """
SELECT SCHEMA_NAME(o.schema_id) AS view_schema, o.name AS view_name,
       d.referenced_entity_name AS depends_on,
       ISNULL(po.type, 'X') AS parent_type
FROM sys.sql_expression_dependencies d
JOIN sys.objects o ON o.object_id = d.referencing_id AND o.type = 'V'
LEFT JOIN sys.objects po ON po.name = d.referenced_entity_name AND po.is_ms_shipped = 0
"""

SQL_COLS_Q = """
SELECT s.name AS schema_name, o.name AS object_name,
       CASE WHEN o.type = 'V' THEN 'VIEW' ELSE 'TABLE' END AS object_kind,
       c.name AS column_name, t.name AS data_type,
       c.max_length, c.precision, c.scale, c.is_nullable, c.column_id
FROM sys.columns c
JOIN sys.objects o ON o.object_id = c.object_id
JOIN sys.schemas s ON s.schema_id = o.schema_id
JOIN sys.types   t ON t.user_type_id = c.user_type_id
WHERE o.type IN ('U','V') AND o.is_ms_shipped = 0
"""


def sql_token():
    for aud in ('https://database.windows.net/', 'https://database.windows.net', 'sql'):
        try:
            t = notebookutils.credentials.getToken(aud)
            if t:
                return t, aud
        except Exception:
            continue
    if KEYVAULT_URL and SP_TENANT_ID:
        try:
            cid = notebookutils.credentials.getSecret(KEYVAULT_URL, KV_SECRET_CLIENT_ID)
            key = notebookutils.credentials.getSecret(KEYVAULT_URL, KV_SECRET_CLIENT_KEY)
            r = requests.post('https://login.microsoftonline.com/' + SP_TENANT_ID + '/oauth2/v2.0/token',
                              data={'grant_type': 'client_credentials', 'client_id': cid,
                                    'client_secret': key,
                                    'scope': 'https://database.windows.net/.default'}, timeout=60)
            if r.ok:
                return r.json()['access_token'], 'service-principal'
        except Exception:
            pass
    return None, None


def sql_query(server, database, query, token):
    import pyodbc
    tok = token.encode('utf-16-le')
    packed = struct.pack('=i', len(tok)) + tok
    conn_str = ('Driver={ODBC Driver 18 for SQL Server};Server=' + server
                + ',1433;Database=' + database
                + ';Encrypt=yes;TrustServerCertificate=no;Connection Timeout=90;')
    # 1256 = SQL_COPT_SS_ACCESS_TOKEN
    with pyodbc.connect(conn_str, attrs_before={1256: packed}) as cn:
        cur = cn.cursor()
        cur.execute(query)
        cols = [d[0] for d in cur.description]
        return [dict(zip(cols, r)) for r in cur.fetchall()]


sql_endpoints = []
for a in (all_items if RUN_SQL_ENDPOINT else []):
    t, wid, iid = a['item'].get('type'), a['ws']['id'], a['item']['id']
    name, wname = a['item'].get('displayName'), a['ws'].get('displayName')
    if t == 'Warehouse':
        d = api(FABRIC_API + '/workspaces/' + wid + '/warehouses/' + iid) or {}
        cs = (d.get('properties') or {}).get('connectionString')
        if cs:
            sql_endpoints.append({'Kind': 'Warehouse', 'Server': cs,
                                  'Database': name, 'WorkspaceName': wname})
    elif t == 'Lakehouse':
        d = api(FABRIC_API + '/workspaces/' + wid + '/lakehouses/' + iid) or {}
        ep = (d.get('properties') or {}).get('sqlEndpointProperties') or {}
        if ep.get('connectionString'):
            sql_endpoints.append({'Kind': 'LakehouseSQLEndpoint', 'Server': ep['connectionString'],
                                  'Database': name, 'WorkspaceName': wname})

obj_rows, dep_rows, sqlcol_rows, unreachable = [], [], [], []

if not RUN_SQL_ENDPOINT:
    print('RUN_SQL_ENDPOINT is False - skipping. You will have no view inventory.')
else:
    print('SQL endpoints discovered: ' + str(len(sql_endpoints)))
    tok, aud = sql_token()
    if not tok:
        print('  No SQL-capable token could be acquired - section skipped')
    elif not sql_endpoints:
        print('  No Warehouses or Lakehouse SQL endpoints found')
    else:
        print('  Token audience: ' + str(aud))
        for ep in sql_endpoints:
            tag = str(ep['WorkspaceName']) + '/' + str(ep['Database'])
            try:
                for r in sql_query(ep['Server'], ep['Database'], SQL_OBJECTS_Q, tok):
                    obj_rows.append(dict(ep, **{k: str(v) for k, v in r.items()}))
                for r in sql_query(ep['Server'], ep['Database'], SQL_DEPS_Q, tok):
                    dep_rows.append(dict(ep, **{k: str(v) for k, v in r.items()}))
                for r in sql_query(ep['Server'], ep['Database'], SQL_COLS_Q, tok):
                    sqlcol_rows.append(dict(ep, **{k: str(v) for k, v in r.items()}))
                print('  ok  ' + tag)
            except Exception as ex:
                unreachable.append({'Endpoint': tag, 'Error': str(ex)[:300]})
                print('  --  ' + tag + '  ' + str(ex)[:120])

obj_df = collect('sql_objects', obj_rows)
dep_df = collect('sql_view_dependencies', dep_rows)
collect('sql_columns', sqlcol_rows)
collect('sql_unreachable', unreachable)

# chain depth: how many view layers a read walks before it reaches a physical table
chain_rows = []
if not dep_df.empty:
    parents, kinds = {}, {}
    for _, r in dep_df.iterrows():
        parents.setdefault((r['WorkspaceName'], r['Database'], r['view_name']), []).append(r['depends_on'])
        kinds[(r['WorkspaceName'], r['Database'], r['depends_on'])] = r['parent_type']

    def depth(ws, db, obj, seen=None):
        seen = (seen or set()) | {(ws, db, obj)}
        best = 0
        for k in parents.get((ws, db, obj), []):
            if (ws, db, k) in seen:
                continue
            if kinds.get((ws, db, k)) == 'V' or (ws, db, k) in parents:
                best = max(best, 1 + depth(ws, db, k, seen))
            else:
                best = max(best, 1)
        return best

    for (ws, db, v) in list(parents.keys()):
        d = depth(ws, db, v)
        chain_rows.append({'WorkspaceName': ws, 'Database': db, 'ViewName': v, 'ChainDepth': d,
                           'Risk': 'HIGH' if d >= 3 else ('MEDIUM' if d == 2 else 'LOW')})

chain_df = collect('sql_view_chains', chain_rows)

if not obj_df.empty:
    display(obj_df.groupby(['WorkspaceName', 'Database', 'object_kind']).size().to_frame('Count'))
if not chain_df.empty:
    display(chain_df.sort_values('ChainDepth', ascending=False).head(25))


## 6b · Detailed lineage — reads, writes and end-to-end paths

Sections 2, 5 and 6 each produce a fragment of lineage. This section resolves them into **directed table-level edges** and then walks them into **end-to-end paths**.

Five extractors, each labelled with its confidence:

| Extractor | Produces | Confidence |
|---|---|---|
| SQL view dependencies (`sys.sql_expression_dependencies`) | view → table/view | **HIGH** — the engine's own answer |
| Notebook code — DataFrame API and SQL strings | table → table, **with direction** | MEDIUM — static parse |
| Pipeline Copy activities — resolved source/sink entities | table → table | MEDIUM |
| Semantic model M expressions | model → table | MEDIUM |
| Shortcuts | item → external or cross-workspace target | **HIGH** |

Then two things nothing else gives you:

- **Column-level lineage for the view layer.** Parses each view's `SELECT` list into output column → source columns. Only covers SQL views, but that is exactly where Gold lives today.
- **End-to-end paths.** For every terminal object — report, semantic model, Gold table — walks upstream to the leaves and emits the full chain with its depth and its weakest link.

> **Be honest about what this is.** Only the SQL and shortcut edges are authoritative. Notebook and pipeline edges are static analysis: they find what the code *references*, not what it *executed*. Dead code appears. Dynamic table names do not. Every edge carries a `Confidence` column — present the HIGH ones as fact and the MEDIUM ones as "this is what the code says, please confirm."

Column-level lineage across Spark transformations is not derivable from any Fabric API. Where a path shows `NOTEBOOK_WRITES` the column trail stops — and that gap, evidenced, is the answer to *"can you show me how this value was derived?"*


In [ ]:
SQL_NOISE = {'SELECT', 'VALUES', 'DUAL', 'LATERAL', 'UNNEST', 'OPENJSON', 'STRING_SPLIT'}
ENTITY_KEYS = ('table', 'tableName', 'entityName', 'objectName', 'fileName',
               'folderPath', 'path', 'container', 'schemaName', 'collection')


def clean_ident(s):
    return str(s).strip().strip('[]`"').strip()


def norm_entity(name):
    n = clean_ident(name)
    if n.lower().startswith('abfss://') or n.lower().startswith('https://'):
        m = re.search(r'/Tables/([^/?]+)', n)
        n = m.group(1) if m else n.rstrip('/').split('/')[-1]
    parts = [p for p in n.split('.') if p]
    if parts:
        n = parts[-1]
    return n.lower()


def parse_sql_lineage(sql):
    if not sql:
        return set(), set()
    t = re.sub(r'--[^\n]*', ' ', str(sql))
    t = re.sub(r'/\*.*?\*/', ' ', t, flags=re.S)
    t = re.sub(r'\s+', ' ', t)
    ctes = set(clean_ident(m.group(1)).upper()
               for m in re.finditer(r'(?:WITH|,)\s+([\w\[\]`"]+)\s+AS\s*\(', t, re.I))
    writes, reads = set(), set()
    write_pats = [r'INSERT\s+(?:INTO|OVERWRITE)\s+(?:TABLE\s+)?([\w\[\]`"\.]+)',
                  r'CREATE\s+(?:OR\s+REPLACE\s+)?(?:TEMPORARY\s+|TEMP\s+|EXTERNAL\s+)*(?:TABLE|VIEW)\s+(?:IF\s+NOT\s+EXISTS\s+)?([\w\[\]`"\.]+)',
                  r'MERGE\s+INTO\s+([\w\[\]`"\.]+)',
                  r'UPDATE\s+([\w\[\]`"\.]+)\s+SET',
                  r'DELETE\s+FROM\s+([\w\[\]`"\.]+)',
                  r'TRUNCATE\s+TABLE\s+([\w\[\]`"\.]+)']
    for p in write_pats:
        for m in re.finditer(p, t, re.I):
            writes.add(clean_ident(m.group(1)))
    for p in [r'FROM\s+([\w\[\]`"\.]+)', r'JOIN\s+([\w\[\]`"\.]+)', r'USING\s+([\w\[\]`"\.]+)']:
        for m in re.finditer(p, t, re.I):
            n = clean_ident(m.group(1))
            if n.upper() in ctes or n.upper() in SQL_NOISE:
                continue
            reads.add(n)
    return reads - writes, writes


def parse_code_lineage(text):
    """Direction-aware. Returns (reads, writes) from Spark / SQL notebook source."""
    reads, writes = set(), set()
    chunks = re.split(r'#\s*CELL\s*\*+', str(text)) or [str(text)]
    for chunk in chunks:
        if re.search(r'%%sql', chunk, re.I):
            r, w = parse_sql_lineage(re.sub(r'%%sql', ' ', chunk, flags=re.I))
            reads |= r
            writes |= w
            continue
        for p in [r'\.saveAsTable\(\s*[frbu]?["\']([^"\']+)["\']',
                  r'\.insertInto\(\s*[frbu]?["\']([^"\']+)["\']',
                  r'DeltaTable\.forName\(\s*\w+\s*,\s*[frbu]?["\']([^"\']+)["\']',
                  r'\.save\(\s*[frbu]?["\']((?:abfss|https)://[^"\']+)["\']',
                  r'DeltaTable\.forPath\(\s*\w+\s*,\s*[frbu]?["\']((?:abfss|https)://[^"\']+)["\']']:
            for m in re.finditer(p, chunk):
                writes.add(m.group(1))
        for p in [r'spark\.read\.table\(\s*[frbu]?["\']([^"\']+)["\']',
                  r'spark\.table\(\s*[frbu]?["\']([^"\']+)["\']',
                  r'\.load\(\s*[frbu]?["\']((?:abfss|https)://[^"\']+)["\']']:
            for m in re.finditer(p, chunk):
                reads.add(m.group(1))
        for m in re.finditer(r'spark\.sql\(\s*[frbu]?("""|\'\'\'|"|\')(.*?)\1', chunk, re.S):
            r, w = parse_sql_lineage(m.group(2))
            reads |= r
            writes |= w
    return reads - writes, writes


def parse_m_lineage(mtext):
    out = set()
    if not mtext:
        return out
    for p in [r'Item\s*=\s*"([^"]+)"', r'\[\s*Name\s*=\s*"([^"]+)"']:
        for m in re.finditer(p, str(mtext)):
            out.add(m.group(1))
    for m in re.finditer(r'Value\.NativeQuery\([^,]+,\s*"((?:[^"\\]|\\.)*)"', str(mtext)):
        r, _ = parse_sql_lineage(m.group(1).replace('\\"', '"'))
        out |= r
    return out


def extract_entities(obj, depth=0):
    found = []
    if depth > 8:
        return found
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in ENTITY_KEYS and isinstance(v, str) and v.strip():
                found.append(v.strip())
            else:
                found.extend(extract_entities(v, depth + 1))
    elif isinstance(obj, list):
        for it in obj:
            found.extend(extract_entities(it, depth + 1))
    return found


def split_top_level(s, sep=','):
    parts, depth, cur = [], 0, ''
    for ch in s:
        if ch in '([':
            depth += 1
        elif ch in ')]':
            depth -= 1
        if ch == sep and depth == 0:
            parts.append(cur)
            cur = ''
        else:
            cur += ch
    parts.append(cur)
    return [p.strip() for p in parts if p.strip()]


def parse_view_columns(sql):
    t = re.sub(r'--[^\n]*', ' ', str(sql))
    t = re.sub(r'/\*.*?\*/', ' ', t, flags=re.S)
    t = re.sub(r'\s+', ' ', t)
    m = re.search(r'\bSELECT\b(.*?)\bFROM\b', t, re.I | re.S)
    if not m:
        return []
    cols = []
    for item in split_top_level(m.group(1)):
        am = re.search(r'\bAS\s+([\w\[\]`"]+)\s*$', item, re.I)
        if am:
            alias = clean_ident(am.group(1))
        else:
            lm = re.search(r'([\w]+)\s*$', item)
            alias = clean_ident(lm.group(1)) if lm else None
        srcs = sorted(set(sm.group(0) for sm in re.finditer(r'\b[\w]+\.[\w]+\b', item)))
        cols.append({'OutputColumn': alias, 'Expression': item[:400],
                     'SourceColumns': '; '.join(srcs), 'SourceColumnCount': len(srcs)})
    return cols


lin_edges, col_edges, viewdefs = [], [], []


def lin_edge(src, tgt, etype, mechanism, confidence, workspace='', detail=''):
    if not src or not tgt:
        return
    lin_edges.append({'SourceEntity': clean_ident(src), 'SourceKey': norm_entity(src),
                      'TargetEntity': clean_ident(tgt), 'TargetKey': norm_entity(tgt),
                      'EdgeType': etype, 'Mechanism': mechanism, 'Confidence': confidence,
                      'Workspace': workspace, 'Detail': str(detail)[:500]})


print('Building detailed lineage')

# ---- 1. SQL view dependencies, definitions and column lineage --------------
SQL_VIEWDEF_Q = """
SELECT SCHEMA_NAME(v.schema_id) AS schema_name, v.name AS view_name, m.definition
FROM sys.views v JOIN sys.sql_modules m ON m.object_id = v.object_id
WHERE v.is_ms_shipped = 0
"""

for _, d in (dep_df.iterrows() if not dep_df.empty else iter([])):
    lin_edge(d['depends_on'], d['view_name'], 'VIEW_READS', 'sys.sql_expression_dependencies',
             'HIGH', str(d['WorkspaceName']), str(d['Database']))

try:
    _tok = tok
except NameError:
    _tok = None
if _tok is None and RUN_SQL_ENDPOINT:
    _tok, _ = sql_token()

if _tok and sql_endpoints:
    for ep in sql_endpoints:
        try:
            for r in sql_query(ep['Server'], ep['Database'], SQL_VIEWDEF_Q, _tok):
                definition = str(r.get('definition') or '')
                viewdefs.append({'WorkspaceName': ep['WorkspaceName'], 'Database': ep['Database'],
                                 'Schema': str(r.get('schema_name')), 'ViewName': str(r.get('view_name')),
                                 'Definition': definition[:60000]})
                for c in parse_view_columns(definition):
                    col_edges.append(dict(c, **{'WorkspaceName': ep['WorkspaceName'],
                                                'Database': ep['Database'],
                                                'ViewName': str(r.get('view_name')),
                                                'Confidence': 'MEDIUM'}))
        except Exception as ex:
            print('  --  view definitions ' + str(ep['Database']) + ': ' + str(ex)[:120])
print('  view edges: ' + str(len(lin_edges)) + '   view column edges: ' + str(len(col_edges)))

# ---- 2. Notebook and Spark code, WITH DIRECTION ----------------------------
code_rows, code_edges = [], 0
for (wname, iname, itype, part), text in DEF_TEXT.items():
    if itype not in ('Notebook', 'SparkJobDefinition'):
        continue
    try:
        reads, writes = parse_code_lineage(text)
    except Exception:
        continue
    code_rows.append({'WorkspaceName': wname, 'ItemName': iname, 'ItemType': itype,
                      'Reads': '; '.join(sorted(reads)[:40]), 'ReadCount': len(reads),
                      'Writes': '; '.join(sorted(writes)[:40]), 'WriteCount': len(writes),
                      'DirectionResolved': bool(writes)})
    for w in writes:
        lin_edge(iname, w, 'NOTEBOOK_WRITES', itype + ': ' + iname, 'MEDIUM', wname, part)
        code_edges += 1
        for rd in reads:
            lin_edge(rd, w, 'DERIVED_FROM', itype + ': ' + iname, 'MEDIUM', wname, 'via ' + iname)
            code_edges += 1
    if not writes:
        for rd in reads:
            lin_edge(rd, iname, 'NOTEBOOK_READS', itype + ': ' + iname, 'LOW', wname, part)
            code_edges += 1

code_df = collect('code_lineage', code_rows)
print('  code edges: ' + str(code_edges))

# ---- 3. Pipeline copy activities -------------------------------------------
pipe_edges = 0
for _, a in (act_df.iterrows() if not act_df.empty else iter([])):
    if str(a.get('ActivityType')) not in ('Copy', 'CopyActivity'):
        continue
    try:
        src_ents = set(extract_entities(json.loads(str(a.get('SourceDetail') or '{}'))))
        snk_ents = set(extract_entities(json.loads(str(a.get('SinkDetail') or '{}'))))
    except Exception:
        continue
    for s in src_ents:
        for t in snk_ents:
            lin_edge(s, t, 'PIPELINE_COPY', str(a['PipelineName']) + ' :: ' + str(a['ActivityName']),
                     'MEDIUM', str(a['WorkspaceName']),
                     str(a.get('SourceType')) + ' -> ' + str(a.get('SinkType')))
            pipe_edges += 1
print('  pipeline copy edges: ' + str(pipe_edges))

# ---- 4. Semantic model M expressions ---------------------------------------
m_edges = 0
for name in ('table_m_source', 'model_expressions'):
    mdf = COLLECTED.get(name, pd.DataFrame())
    for _, e in (mdf.iterrows() if not mdf.empty else iter([])):
        for tbl in parse_m_lineage(e.get('Expression')):
            lin_edge(tbl, str(e.get('DatasetName')), 'MODEL_SOURCE', 'Power Query M',
                     'MEDIUM', str(e.get('WorkspaceName')), str(e.get('TableName', '')))
            m_edges += 1
print('  model source edges: ' + str(m_edges))

# ---- 5. Shortcuts and 6. Scanner BI lineage --------------------------------
for _, s in (sc_df.iterrows() if not sc_df.empty else iter([])):
    lin_edge(str(s['TargetDetail']), str(s['HostItem']) + '/' + str(s['ShortcutName']),
             'SHORTCUT_TO', 'OneLake shortcut', 'HIGH', str(s['WorkspaceName']), str(s['TargetType']))

_bi = COLLECTED.get('lineage', pd.DataFrame())
for _, l in (_bi.iterrows() if not _bi.empty else iter([])):
    lin_edge(str(l['UpstreamId']), str(l['Downstream']), 'BI_DEPENDS_ON', 'Scanner API',
             'HIGH', str(l['WorkspaceName']), str(l['UpstreamType']))

lineage_df = collect('lineage_edges', lin_edges)
collect('lineage_column_view', col_edges)
collect('sql_view_definitions', viewdefs)

# ---- 7. Walk end-to-end paths ----------------------------------------------
paths = []
if not lineage_df.empty:
    upstream, all_sources = {}, set()
    for _, e in lineage_df.iterrows():
        upstream.setdefault(e['TargetKey'], []).append((e['SourceKey'], e['EdgeType'], e['Confidence']))
        all_sources.add(e['SourceKey'])
    terminals = sorted(set(upstream.keys()) - all_sources)

    def walk(node, trail, conf, seen, depth=0):
        if depth > 12 or node in seen:
            paths.append({'Terminal': trail[0], 'Path': ' <- '.join(trail), 'Depth': len(trail) - 1,
                          'WeakestLink': conf, 'Truncated': True})
            return
        ups = upstream.get(node)
        if not ups:
            paths.append({'Terminal': trail[0], 'Path': ' <- '.join(trail), 'Depth': len(trail) - 1,
                          'WeakestLink': conf, 'Truncated': False})
            return
        rank = {'HIGH': 0, 'MEDIUM': 1, 'LOW': 2}
        for (src, etype, c) in ups[:6]:
            worst = conf if rank[conf] >= rank[c] else c
            walk(src, trail + [src + ' [' + etype + ']'], worst, seen | {node}, depth + 1)

    for t in terminals[:400]:
        walk(t, [t], 'HIGH', set())

paths_df = collect('lineage_paths', paths)

if not lineage_df.empty:
    display(lineage_df.groupby(['EdgeType', 'Confidence']).size().to_frame('Edges'))
if not paths_df.empty:
    display(paths_df.sort_values('Depth', ascending=False).head(25))
    print()
    print('Longest chain: ' + str(int(paths_df['Depth'].max())) + ' hops')
    inferred = int((paths_df['WeakestLink'] != 'HIGH').sum())
    print('Paths whose weakest link is inferred: ' + str(inferred) + ' of ' + str(len(paths_df)))
    print('Those are what the code says, not what was observed. Confirm before presenting.')


## 7 · Estate graph

Every object and every relationship as a node/edge list, plus Graphviz source written to `Files/estate-graph.dot`.

Edge types: `ASSIGNED_TO_CAPACITY` · `IN_WORKSPACE` · `TABLE_OF` · `SHORTCUT_TO` · `DEPENDS_ON` · `MODEL_RELATIONSHIP` · `PIPELINE_INVOKES` · `VIEW_READS`

`graph_nodes` and `graph_edges` load directly into Power BI, networkx, Neo4j or Gephi. To render the DOT file, download it and run `dot -Tsvg estate-graph.dot -o estate.svg`.


In [ ]:
nodes, edges, seen_nodes = [], [], set()


def rows(df):
    return df.iterrows() if isinstance(df, pd.DataFrame) and not df.empty else iter([])


def node(nid, label, ntype, workspace='', extra=''):
    if not nid or nid in seen_nodes:
        return
    seen_nodes.add(nid)
    nodes.append({'NodeId': nid, 'Label': str(label), 'NodeType': ntype,
                  'Workspace': workspace, 'Extra': extra})


def edge(src, tgt, etype, detail=''):
    if src and tgt:
        edges.append({'SourceId': src, 'TargetId': tgt, 'EdgeType': etype, 'Detail': detail})


for _, c in rows(cap_df):
    node('cap:' + str(c['CapacityId']), c['DisplayName'], 'Capacity', extra=str(c['Sku']))

for _, w in rows(ws_df):
    node('ws:' + str(w['WorkspaceId']), w['DisplayName'], 'Workspace', extra=str(w['CapacitySku']))
    edge('ws:' + str(w['WorkspaceId']), 'cap:' + str(w['CapacityId']), 'ASSIGNED_TO_CAPACITY')

for _, i in rows(items_df):
    node('item:' + str(i['ItemId']), i['ItemName'], str(i['ItemType']), str(i['WorkspaceName']))
    edge('item:' + str(i['ItemId']), 'ws:' + str(i['WorkspaceId']), 'IN_WORKSPACE')

for _, t in rows(tables_df):
    tid = 'tbl:' + str(t['LakehouseId']) + ':' + str(t['TableName'])
    node(tid, t['TableName'], 'Table', str(t['WorkspaceName']), str(t['TableType']))
    edge(tid, 'item:' + str(t['LakehouseId']), 'TABLE_OF')

for _, s in rows(sc_df):
    sid = 'sc:' + str(s['HostItem']) + ':' + str(s['ShortcutName'])
    node(sid, s['ShortcutName'], 'Shortcut', str(s['WorkspaceName']), str(s['ShortcutPath']))
    detail = str(s['TargetDetail'])
    if s['TargetType'] == 'OneLake' and 'item=' in detail:
        tgt = 'item:' + detail.split('item=')[1].split(';')[0]
    else:
        tgt = 'ext:' + str(s['TargetType']) + ':' + detail
        node(tgt, detail[:80], 'External:' + str(s['TargetType']), extra=str(s['TargetType']))
    edge(sid, tgt, 'SHORTCUT_TO', str(s['TargetType']))

for _, d in rows(COLLECTED.get('datasources', pd.DataFrame())):
    node('src:' + str(d['DatasourceId']),
         str(d['DatasourceType']) + ': ' + str(d['Server']) + '/' + str(d['Database']),
         'Datasource', extra=str(d['DatasourceType']))

for _, l in rows(COLLECTED.get('lineage', pd.DataFrame())):
    prefix = 'src:' if str(l['UpstreamType']).startswith('Datasource') else 'item:'
    edge('item:' + str(l['DownstreamId']), prefix + str(l['UpstreamId']), 'DEPENDS_ON', str(l['UpstreamType']))

# resolved table-level lineage from section 6b, direction included
for _, e in rows(COLLECTED.get('lineage_edges', pd.DataFrame())):
    s_id = 'ent:' + str(e['SourceKey'])
    t_id = 'ent:' + str(e['TargetKey'])
    node(s_id, e['SourceEntity'], 'Entity', str(e['Workspace']))
    node(t_id, e['TargetEntity'], 'Entity', str(e['Workspace']))
    edge(s_id, t_id, str(e['EdgeType']), str(e['Confidence']) + ' · ' + str(e['Mechanism'])[:80])

for _, r in rows(COLLECTED.get('model_relationships', pd.DataFrame())):
    mid = 'mt:' + str(r['WorkspaceName']) + ':' + str(r['DatasetName']) + ':'
    node(mid + str(r['FromTable']), r['FromTable'], 'ModelTable', str(r['WorkspaceName']), str(r['DatasetName']))
    node(mid + str(r['ToTable']), r['ToTable'], 'ModelTable', str(r['WorkspaceName']), str(r['DatasetName']))
    edge(mid + str(r['FromTable']), mid + str(r['ToTable']), 'MODEL_RELATIONSHIP',
         str(r['FromColumn']) + ' -> ' + str(r['ToColumn']) + ' [' + str(r['CrossFilter']) + ']')

for _, a in rows(act_df):
    if a['Invokes']:
        pid = 'pipe:' + str(a['PipelineName'])
        node(pid, a['PipelineName'], 'DataPipeline', str(a['WorkspaceName']))
        edge(pid, str(a['Invokes']), 'PIPELINE_INVOKES', str(a['ActivityType']))

for _, v in rows(dep_df):
    vid = 'sqlobj:' + str(v['Database']) + ':' + str(v['view_name'])
    pid = 'sqlobj:' + str(v['Database']) + ':' + str(v['depends_on'])
    node(vid, v['view_name'], 'SqlView', str(v['WorkspaceName']), str(v['Database']))
    node(pid, v['depends_on'], 'SqlTable' if v['parent_type'] == 'U' else 'SqlView',
         str(v['WorkspaceName']), str(v['Database']))
    edge(vid, pid, 'VIEW_READS', str(v['parent_type']))

nodes_df = collect('graph_nodes', nodes)
edges_df = collect('graph_edges', edges)

PALETTE = {'Capacity': '#1A7A1A', 'Workspace': '#EDEAE3', 'Lakehouse': '#D4FF3F',
           'Warehouse': '#D4FF3F', 'SemanticModel': '#9FD3FF', 'Report': '#FFD79F',
           'DataPipeline': '#FFB3B3', 'Notebook': '#E3D0FF', 'Table': '#FFFFFF',
           'Datasource': '#FF9F9F', 'Shortcut': '#FFF2B3', 'SqlView': '#FF7F7F',
           'SqlTable': '#FFFFFF', 'ModelTable': '#FFFFFF', 'Entity': '#EAF3EA'}

lines = ['digraph FabricEstate {',
         '  rankdir=LR; node [shape=box,style=filled,fontname="Helvetica"]; edge [fontsize=8];']
idmap = {}
for n, nd in enumerate(nodes):
    idmap[nd['NodeId']] = 'n' + str(n)
    default = '#FF9F9F' if str(nd['NodeType']).startswith('External') else '#FFFFFF'
    fill = PALETTE.get(nd['NodeType'], default)
    label = nd['Label'].replace('"', "'")
    lines.append('  n' + str(n) + ' [label="' + label + ' [' + str(nd['NodeType'])
                 + ']",fillcolor="' + fill + '"];')
for e in edges:
    if e['SourceId'] in idmap and e['TargetId'] in idmap:
        if e['EdgeType'] == 'SHORTCUT_TO':
            style = 'dashed'
        elif str(e['Detail']).startswith('MEDIUM') or str(e['Detail']).startswith('LOW'):
            style = 'dotted'
        else:
            style = 'solid'
        lines.append('  ' + idmap[e['SourceId']] + ' -> ' + idmap[e['TargetId']]
                     + ' [label="' + e['EdgeType'] + '",style=' + style + '];')
lines.append('}')
DOT = '\n'.join(lines)

try:
    notebookutils.fs.put('Files/estate-graph.dot', DOT, True)
    print('Graphviz source written to Files/estate-graph.dot')
except Exception as ex:
    print('Could not write DOT file: ' + str(ex)[:200])

print('Graph: ' + str(len(nodes)) + ' nodes, ' + str(len(edges)) + ' edges')
print('Dotted edges are inferred (MEDIUM/LOW confidence), solid are authoritative.')
if not edges_df.empty:
    display(edges_df.groupby('EdgeType').size().sort_values(ascending=False).to_frame('Count'))


## 8 · Findings

Every row is a **hypothesis to test**, not a conclusion. The `AskInSession` column is the point — a finding without a question is an assertion you have not earned.

Present each as *"what we observed → what it implies if it holds → what we would need to confirm it."* Never drop the third clause; it is what turns an accusation into a request and keeps the room cooperating rather than defending.


In [ ]:
findings = []


def finding(sev, area, what, evidence, question):
    findings.append({'Severity': sev, 'Area': area, 'Finding': what,
                     'Evidence': evidence, 'AskInSession': question})


# --- views vs persisted tables -------------------------------------------
if not obj_df.empty:
    for (wn, db), grp in obj_df.groupby(['WorkspaceName', 'Database']):
        views = int((grp['object_kind'] == 'VIEW').sum())
        tabs = int((grp['object_kind'] == 'TABLE').sum())
        if views and views >= tabs:
            finding('HIGH', 'Gold persistence',
                    str(db) + ' exposes ' + str(views) + ' views against ' + str(tabs) + ' persisted tables',
                    str(wn) + '/' + str(db),
                    'Which of these are consumed by sponsors, and at what concurrency?')

if not chain_df.empty:
    for _, c in chain_df[chain_df['ChainDepth'] >= 2].iterrows():
        finding('HIGH' if c['ChainDepth'] >= 3 else 'MEDIUM', 'View chain',
                "View '" + str(c['ViewName']) + "' sits " + str(c['ChainDepth'])
                + ' layers above a physical table',
                str(c['WorkspaceName']) + '/' + str(c['Database']),
                'Every read recomputes this chain. What is the read volume on it?')

# --- metric canon ---------------------------------------------------------
meas = COLLECTED.get('model_measures', pd.DataFrame())
if not meas.empty:
    for name, grp in meas.groupby('MeasureName'):
        variants = grp['Expression'].nunique()
        if len(grp) > 1 and variants > 1:
            where = ' | '.join((grp['WorkspaceName'] + '/' + grp['DatasetName']).tolist())
            finding('HIGH', 'Metric canon',
                    "Measure '" + str(name) + "' has " + str(variants) + ' different definitions',
                    where, 'Which definition is authoritative, and who owns it?')

# --- capacity isolation ---------------------------------------------------
if not ws_df.empty:
    etl_kw = ('bronze', 'silver', 'ingest', 'land', 'conform', 'raw')
    srv_kw = ('gold', 'report', 'sponsor', 'semantic', 'serve', 'analytic')
    for cid, grp in ws_df[ws_df['CapacityId'].notna()].groupby('CapacityId'):
        names = grp['DisplayName'].fillna('').tolist()
        has_etl = any(any(k in n.lower() for k in etl_kw) for n in names)
        has_srv = any(any(k in n.lower() for k in srv_kw) for n in names)
        if has_etl and has_srv:
            finding('HIGH', 'Capacity isolation',
                    'Capacity ' + str(grp['CapacityName'].iloc[0]) + ' (' + str(grp['CapacitySku'].iloc[0])
                    + ') hosts both transformation and serving workspaces',
                    ' | '.join(names), 'Which workload caused the last throttling event?')

# --- Delta maintenance ----------------------------------------------------
if not delta_df.empty and 'MaintenancePattern' in delta_df.columns:
    for _, r in delta_df[(delta_df['MaintenancePattern'] == 'FULL REBUILD (Overwrite)')
                         & (delta_df['SizeMB'] > 500)].iterrows():
        finding('HIGH', 'Maintenance pattern',
                "Table '" + str(r['TableName']) + "' (" + str(r['SizeMB'])
                + ' MB) is fully rewritten, not merged',
                str(r['WorkspaceName']) + '/' + str(r['LakehouseName']),
                'Why a full rebuild rather than an incremental MERGE?')

    for _, r in delta_df[delta_df['SmallFileRisk'] == True].iterrows():
        finding('MEDIUM', 'Small files',
                "Table '" + str(r['TableName']) + "': " + str(r['NumFiles'])
                + ' files averaging ' + str(r['AvgFileMB']) + ' MB',
                str(r['WorkspaceName']) + '/' + str(r['LakehouseName']),
                'Who owns OPTIMIZE and VACUUM scheduling for this lakehouse?')

    for _, r in delta_df[delta_df['LastWriteUtc'].notna()].iterrows():
        try:
            ts = pd.to_datetime(r['LastWriteUtc']).to_pydatetime().replace(tzinfo=None)
            age = (dt.datetime.utcnow() - ts).days
        except Exception:
            continue
        if age > 90:
            finding('LOW', 'Estate hygiene',
                    "Table '" + str(r['TableName']) + "' not written for " + str(age) + ' days',
                    str(r['WorkspaceName']) + '/' + str(r['LakehouseName']),
                    'Still in use, or residue from the migration?')

# --- semantic models ------------------------------------------------------
mdl = COLLECTED.get('semantic_models', pd.DataFrame())
if not mdl.empty:
    for _, m in mdl[(mdl['RelationshipCount'] == 0) & (mdl['TableCount'] > 1)].iterrows():
        finding('MEDIUM', 'Data model',
                "Model '" + str(m['DatasetName']) + "' has " + str(m['TableCount'])
                + ' tables and no relationships',
                str(m['WorkspaceName']), 'Is this a flat extract? Where do the joins live?')

    rls_df = COLLECTED.get('rls_roles', pd.DataFrame())
    rls_names = set(rls_df['DatasetName']) if not rls_df.empty else set()
    for _, m in mdl.iterrows():
        wn = str(m['WorkspaceName']).lower()
        if any(k in wn for k in ('sponsor', 'share', 'external', 'client')) and m['DatasetName'] not in rls_names:
            finding('HIGH', 'Security',
                    "Sponsor-facing model '" + str(m['DatasetName']) + "' has no RLS roles",
                    str(m['WorkspaceName']),
                    "What prevents one sponsor seeing another sponsor's data?")

# --- shortcuts, orchestration, change control -----------------------------
if not sc_df.empty:
    for _, s in sc_df[sc_df['IsExternal'] == True].iterrows():
        finding('MEDIUM', 'Data residency',
                'External shortcut to ' + str(s['TargetType']) + ' outside OneLake',
                str(s['WorkspaceName']) + '/' + str(s['HostItem']) + ' -> ' + str(s['TargetDetail']),
                'Who owns this store, and is it in scope for residency and access review?')

if not act_df.empty:
    no_dep = act_df[(act_df['DependsOn'] == '') & (act_df['ActivityType'] == 'Copy')]
    if len(no_dep) > 10:
        finding('LOW', 'Orchestration',
                str(len(no_dep)) + ' copy activities have no declared dependency',
                'across ' + str(no_dep['PipelineName'].nunique()) + ' pipelines',
                'Is ordering implicit? What happens when one runs long?')

if not items_df.empty:
    scheduled = set(sched_df['ItemName']) if not sched_df.empty else set()
    for _, p in items_df[items_df['ItemType'] == 'DataPipeline'].iterrows():
        if p['ItemName'] not in scheduled:
            finding('LOW', 'Orchestration',
                    "Pipeline '" + str(p['ItemName']) + "' has no schedule",
                    str(p['WorkspaceName']), 'Triggered by another pipeline, run manually, or dead?')

git_names = set(git_df['WorkspaceName']) if not git_df.empty else set()
if not ws_df.empty:
    for _, w in ws_df[ws_df['CapacityId'].notna()].iterrows():
        if w['DisplayName'] not in git_names:
            finding('MEDIUM', 'Change control',
                    "Workspace '" + str(w['DisplayName']) + "' is not connected to Git",
                    str(w['WorkspaceId']), 'How are changes here reviewed and promoted?')

find_df = pd.DataFrame(findings)
if not find_df.empty:
    order = {'HIGH': 0, 'MEDIUM': 1, 'LOW': 2}
    find_df = (find_df.assign(_o=find_df['Severity'].map(order))
                      .sort_values(['_o', 'Area']).drop(columns='_o'))
COLLECTED['findings'] = find_df

print('Findings: ' + str(len(find_df)))
if not find_df.empty:
    display(find_df.groupby(['Severity', 'Area']).size().to_frame('Count'))
    display(find_df[find_df['Severity'] == 'HIGH'].head(30))


In [ ]:
if not cap_metrics_df.empty:
    thr = cap_metrics_df[cap_metrics_df['Metric'] == 'ThrottlingEvent']
    for cname, grp in thr.groupby('CapacityName'):
        finding('HIGH', 'Capacity throttling',
                'Capacity ' + str(cname) + ' throttled on ' + str(len(grp)) + ' measured time points',
                str(grp['Detail'].iloc[0])[:200],
                'Which workload caused these, and was a sponsor-facing read affected?')

    top = cap_metrics_df[cap_metrics_df['Metric'] == 'TopConsumingItem'].head(5)
    if not top.empty:
        finding('MEDIUM', 'Capacity consumption',
                'Top 5 consuming items account for the largest share of CU',
                ' | '.join(top['Detail'].astype(str).str[:80].tolist()),
                'Is this list reviewed, and by whom?')
else:
    finding('MEDIUM', 'Evidence gap', 'No capacity metrics captured',
            'Capacity Metrics app not reachable',
            'Can we get 60 days of Capacity Metrics exported before the session?')

find_df = pd.DataFrame(findings)
if not find_df.empty:
    order = {'HIGH': 0, 'MEDIUM': 1, 'LOW': 2}
    find_df = (find_df.assign(_o=find_df['Severity'].map(order))
                      .sort_values(['_o', 'Area']).drop(columns='_o'))
COLLECTED['findings'] = find_df

print('Findings after capacity pass: ' + str(len(find_df)))
if not find_df.empty:
    display(find_df[find_df['Severity'] == 'HIGH'].head(30))


---

# PART TWO · Data content

**Everything above reads metadata only. Everything below reads rows.**

Sections 9–11 open clinical data. At a regulated CRO that is a different permission posture, a different approval, and potentially PHI in scope.

> **`READ_DATA_CONTENT` defaults to `False`.** Part One runs on its own under a light approval. Set the flag to `True` only when data access has been agreed. Nothing is ever written back to a source table — output goes only to `estate_dq_*` tables in the attached Lakehouse.

| § | Delivery script | Session | Reads data? |
|---|---|---|---|
| 9 | `04_profile_entities` — six quality dimensions | S03 | Yes |
| 10 | `05_identifier_reconciliation` — Study / Site / Subject | S02 | Yes |
| 11 | `10_great_expectations` — validation suite | S03 | Yes |
| 12 | `08_semantic_candidate` — TMDL model | S05 | Schema + row counts |
| 13 | `09_ontology_binding` — Fabric IQ spec | S05 | No |

Cell 26 holds all Part Two configuration. **Auto-discovery works with no config** — you get completeness, uniqueness and accuracy on every table. Adding `RULES` unlocks validity, consistency and timeliness.


In [ ]:
import uuid
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

# ---- master switch for everything below ------------------------------------
READ_DATA_CONTENT    = False   # set True only when data access has been approved

RUN_PROFILING        = True    # 04
RUN_RECONCILIATION   = True    # 05
RUN_EXPECTATIONS     = True    # 10
RUN_SEMANTIC_MODEL   = True    # 08
RUN_ONTOLOGY         = True    # 09

# ---- scope -----------------------------------------------------------------
AUTO_DISCOVER        = True
PROFILE_TABLES       = []      # explicit list overrides discovery
TABLE_NAME_FILTER    = ''
MAX_PROFILE_TABLES   = 200
SAMPLE_FRACTION      = 0.0     # 0 = full scan; 0.1 = 10% sample for very large tables
GOLD_PREFIXES        = ('fct_', 'dim_', 'gold_')

# ---- domain rules ----------------------------------------------------------
# REPLACE these with real table and column names from the Part One output.
# Every key is optional. Tables with no entry still get completeness,
# uniqueness and accuracy. Rules add validity, consistency and timeliness.
RULES = {
    'slv_subject': {
        'business_key': ['study_id', 'usubjid'],
        'not_null': ['study_id', 'usubjid', 'site_id'],
        'allowed_values': {'sex': ['M', 'F', 'U'],
                           'subject_status': ['SCREENED', 'ENROLLED', 'WITHDRAWN', 'COMPLETED']},
        'ranges': {'age': [0, 120]},
        'patterns': {'usubjid': r'^[A-Za-z0-9\-_]+$'},
        'timeliness_column': '_ingest_ts',
        'freshness_sla_hours': 24,
        'references': [{'column': 'site_id', 'ref_table': 'slv_site', 'ref_column': 'site_id'}],
        'date_order': [['informed_consent_date', 'first_dose_date']],
        'expected_row_range': [1, 5000000],
    },
    'slv_adverse_event': {
        'business_key': ['study_id', 'usubjid', 'ae_seq'],
        'not_null': ['study_id', 'usubjid', 'ae_term', 'ae_start_date'],
        'allowed_values': {'ae_serious': ['Y', 'N'],
                           'ae_severity': ['MILD', 'MODERATE', 'SEVERE']},
        'timeliness_column': '_ingest_ts',
        'freshness_sla_hours': 24,
        'references': [{'column': 'usubjid', 'ref_table': 'slv_subject', 'ref_column': 'usubjid'}],
        'date_order': [['ae_start_date', 'ae_end_date']],
    },
    'slv_site': {
        'business_key': ['site_id'],
        'not_null': ['site_id', 'study_id', 'country'],
        'timeliness_column': '_ingest_ts',
        'freshness_sla_hours': 24,
    },
}

# Values that look populated but carry no meaning. Counted under Accuracy.
SUSPICIOUS_DEFAULTS = ['', 'NA', 'N/A', 'NULL', 'NONE', 'UNKNOWN', 'UNK', 'TBD',
                       '9999', '-1', '1900-01-01', '1901-01-01', '0001-01-01']

# ---- identifier reconciliation (05) ----------------------------------------
IDENTIFIER_MAP = {
    'Study': [
        {'source': 'Veeva EDC',     'table': 'brz_edc_subjects', 'column': 'study_id'},
        {'source': 'Medidata Rave', 'table': 'brz_rave_subject', 'column': 'studyoid'},
        {'source': 'Veeva CTMS',    'table': 'brz_ucv_study',    'column': 'study_number'},
    ],
    'Site': [
        {'source': 'Veeva EDC',     'table': 'brz_edc_sites',    'column': 'site_id'},
        {'source': 'Veeva CTMS',    'table': 'brz_ucv_site',     'column': 'site_number'},
    ],
    'Subject': [
        {'source': 'Veeva EDC',     'table': 'brz_edc_subjects', 'column': 'usubjid'},
        {'source': 'Medidata Rave', 'table': 'brz_rave_subject', 'column': 'subjectkey'},
        {'source': 'Veeva CTMS',    'table': 'brz_ucv_subject',  'column': 'subject_id'},
    ],
}
RECON_NORMALISE = {'trim': True, 'upper': True, 'strip_leading_zeros': False}

# ---- clinical concepts for the ontology (09) -------------------------------
CONCEPT_MAP = {
    'dim_study': 'Study', 'dim_site': 'Site', 'dim_investigator': 'Investigator',
    'dim_subject': 'Subject', 'dim_visit': 'Visit', 'dim_country': 'Country',
    'fct_adverse_event': 'AdverseEvent', 'fct_subject_visit': 'SubjectVisit',
    'fct_enrolment_daily': 'EnrolmentSnapshot', 'fct_query_lifecycle': 'DataQuery',
    'fct_protocol_deviation': 'ProtocolDeviation', 'fct_lab_result': 'LabResult',
}
ONTOLOGY_NAME = 'ClinicalTrialOntology'


# ---- helpers ---------------------------------------------------------------
def read_table(name):
    try:
        df = spark.table(name)
    except AnalysisException:
        return None
    except Exception:
        return None
    if SAMPLE_FRACTION and 0 < SAMPLE_FRACTION < 1:
        df = df.sample(False, SAMPLE_FRACTION, seed=42)
    return df


def discover_tables():
    if PROFILE_TABLES:
        return list(PROFILE_TABLES)
    if not AUTO_DISCOVER:
        return []
    try:
        names = [r['tableName'] for r in spark.sql('SHOW TABLES').collect()]
    except Exception:
        return []
    names = [n for n in names if not n.startswith(TABLE_PREFIX)]
    if TABLE_NAME_FILTER:
        names = [n for n in names if TABLE_NAME_FILTER.lower() in n.lower()]
    return names[:MAX_PROFILE_TABLES]


def write_file(rel_path, content):
    try:
        notebookutils.fs.put('Files/' + rel_path, content, True)
        print('  wrote Files/' + rel_path)
        return True
    except Exception as ex:
        print('  could not write Files/' + rel_path + ': ' + str(ex)[:180])
        return False


DATA_TABLES = discover_tables() if READ_DATA_CONTENT else []

if not READ_DATA_CONTENT:
    print('READ_DATA_CONTENT is False - Part Two will not open any data.')
    print('Set it to True once data access is approved.')
else:
    print(str(len(DATA_TABLES)) + ' tables in scope for profiling')
    print('  ' + ', '.join(DATA_TABLES[:20]) + ('  ...' if len(DATA_TABLES) > 20 else ''))


## 9 · Profile against the six dimensions
### `04_profile_entities` · Session S03

| Dimension | Measured how | Needs rules? |
|---|---|---|
| **Completeness** | Null and blank rate per column; declared mandatory columns | No |
| **Uniqueness** | Duplicate rate on the business key; cardinality per column | Key needed |
| **Validity** | Value in allowed set, within range, matching pattern | Yes |
| **Timeliness** | Age of the newest record against a freshness SLA | Column needed |
| **Consistency** | Referential orphans; date ordering; row-count plausibility | Yes |
| **Accuracy** | Placeholder values that look populated but mean nothing | No |

Accuracy is the honest weak spot. Without a trusted reference you cannot prove a value is *correct*, only that it is *plausible*. This measures the placeholder proxy — `UNKNOWN`, `1900-01-01`, `N/A`. Section 10 adds cross-source agreement, which is the strongest accuracy signal available without a gold standard.

Output is one row per rule evaluated, with a pass rate and a severity.


In [ ]:
prof_results, col_stats = [], []


def dq_result(table, dimension, rule, scope, total, failed, severity='MEDIUM', detail=''):
    passed = max(total - failed, 0)
    prof_results.append({'Table': table, 'Dimension': dimension, 'Rule': rule, 'Scope': scope,
                         'TotalRows': total, 'FailedRows': failed, 'PassedRows': passed,
                         'PassRate': round(100.0 * passed / total, 3) if total else None,
                         'Severity': severity, 'Detail': detail})


def blank_expr(col, dtype):
    c = F.col('`' + col + '`')
    return (c.isNull() | (F.trim(c) == '')) if dtype in ('string', 'varchar', 'char') else c.isNull()


def profile_table(name):
    df = read_table(name)
    if df is None:
        dq_result(name, 'Completeness', 'table_readable', name, 0, 1, 'HIGH', 'table not readable')
        return
    cols = [(f.name, f.dataType.simpleString()) for f in df.schema.fields]
    coltypes = dict(cols)
    total = df.count()
    rules = RULES.get(name, {})
    if total == 0:
        dq_result(name, 'Completeness', 'table_not_empty', name, 1, 1, 'HIGH', 'zero rows')
        return

    nulls = df.agg(*[F.count(F.when(blank_expr(c, t), F.lit(1))).alias(c) for c, t in cols]).collect()[0].asDict()
    mandatory = set(rules.get('not_null', []))
    for c, t in cols:
        n = int(nulls.get(c) or 0)
        sev = 'HIGH' if c in mandatory and n > 0 else ('LOW' if n == 0 else 'MEDIUM')
        dq_result(name, 'Completeness',
                  'mandatory_not_null' if c in mandatory else 'null_or_blank_rate', c, total, n, sev)

    card = df.agg(*[F.approx_count_distinct(F.col('`' + c + '`')).alias(c) for c, _ in cols]).collect()[0].asDict()
    for c, t in cols:
        d = int(card.get(c) or 0)
        col_stats.append({'Table': name, 'Column': c, 'DataType': t, 'TotalRows': total,
                          'NullOrBlank': int(nulls.get(c) or 0),
                          'NullPct': round(100.0 * (nulls.get(c) or 0) / total, 3),
                          'ApproxDistinct': d,
                          'DistinctPct': round(100.0 * d / total, 3) if total else None})

    bk = rules.get('business_key')
    if bk and all(k in coltypes for k in bk):
        dupes = (df.groupBy(*[F.col('`' + k + '`') for k in bk]).count()
                   .filter(F.col('count') > 1)
                   .agg(F.coalesce(F.sum(F.col('count') - 1), F.lit(0))).collect()[0][0])
        dq_result(name, 'Uniqueness', 'business_key_unique', ' + '.join(bk), total, int(dupes or 0), 'HIGH')

    for c, allowed in (rules.get('allowed_values') or {}).items():
        if c in coltypes:
            bad = df.filter(F.col('`' + c + '`').isNotNull()
                            & ~F.upper(F.trim(F.col('`' + c + '`').cast('string')))
                            .isin([str(a).upper() for a in allowed])).count()
            dq_result(name, 'Validity', 'value_in_allowed_set', c, total, bad, 'HIGH',
                      'allowed: ' + ', '.join([str(a) for a in allowed]))
    for c, rng in (rules.get('ranges') or {}).items():
        if c in coltypes:
            bad = df.filter(F.col('`' + c + '`').isNotNull() & ~F.col('`' + c + '`').between(rng[0], rng[1])).count()
            dq_result(name, 'Validity', 'value_in_range', c, total, bad, 'MEDIUM', 'range: ' + str(rng))
    for c, pat in (rules.get('patterns') or {}).items():
        if c in coltypes:
            bad = df.filter(F.col('`' + c + '`').isNotNull()
                            & ~F.col('`' + c + '`').cast('string').rlike(pat)).count()
            dq_result(name, 'Validity', 'value_matches_pattern', c, total, bad, 'MEDIUM', 'pattern: ' + pat)

    tcol = rules.get('timeliness_column')
    if tcol and tcol in coltypes:
        newest = df.agg(F.max(F.col('`' + tcol + '`'))).collect()[0][0]
        if newest is not None:
            try:
                ts = pd.to_datetime(str(newest)).to_pydatetime().replace(tzinfo=None)
                lag_h = round((dt.datetime.utcnow() - ts).total_seconds() / 3600.0, 2)
            except Exception:
                lag_h = None
            sla = rules.get('freshness_sla_hours')
            breach = 1 if (lag_h is not None and sla and lag_h > sla) else 0
            dq_result(name, 'Timeliness', 'freshness_within_sla', tcol, 1, breach,
                      'HIGH' if breach else 'LOW',
                      'newest=' + str(newest) + ' lag_hours=' + str(lag_h) + ' sla=' + str(sla))

    for ref in (rules.get('references') or []):
        c, rt, rc = ref['column'], ref['ref_table'], ref['ref_column']
        rdf = read_table(rt)
        if rdf is None or c not in coltypes:
            dq_result(name, 'Consistency', 'referential_integrity', c, total, 0, 'LOW',
                      'reference table ' + rt + ' not readable')
            continue
        keys = rdf.select(F.col('`' + rc + '`').alias('__k')).distinct()
        orphans = (df.filter(F.col('`' + c + '`').isNotNull())
                     .join(keys, df['`' + c + '`'] == keys['__k'], 'left_anti').count())
        dq_result(name, 'Consistency', 'referential_integrity',
                  c + ' -> ' + rt + '.' + rc, total, orphans, 'HIGH')
    for pair in (rules.get('date_order') or []):
        a, b = pair[0], pair[1]
        if a in coltypes and b in coltypes:
            bad = df.filter(F.col('`' + a + '`').isNotNull() & F.col('`' + b + '`').isNotNull()
                            & (F.col('`' + a + '`') > F.col('`' + b + '`'))).count()
            dq_result(name, 'Consistency', 'date_order', a + ' <= ' + b, total, bad, 'HIGH')
    rr = rules.get('expected_row_range')
    if rr:
        breach = 0 if rr[0] <= total <= rr[1] else 1
        dq_result(name, 'Consistency', 'row_count_plausible', name, 1, breach,
                  'MEDIUM' if breach else 'LOW', 'actual=' + str(total) + ' expected=' + str(rr))

    strcols = [c for c, t in cols if t in ('string', 'varchar', 'char')]
    if strcols:
        sus = [s.upper() for s in SUSPICIOUS_DEFAULTS]
        hits = df.agg(*[F.count(F.when(F.upper(F.trim(F.col('`' + c + '`'))).isin(sus), F.lit(1))).alias(c)
                        for c in strcols]).collect()[0].asDict()
        for c in strcols:
            n = int(hits.get(c) or 0)
            if n:
                dq_result(name, 'Accuracy', 'placeholder_values', c, total, n, 'MEDIUM',
                          'values that look populated but carry no meaning')


if READ_DATA_CONTENT and RUN_PROFILING:
    for n, t in enumerate(DATA_TABLES, start=1):
        print('[' + str(n) + '/' + str(len(DATA_TABLES)) + '] ' + t)
        try:
            profile_table(t)
        except Exception as ex:
            dq_result(t, 'Completeness', 'profile_error', t, 0, 1, 'HIGH', str(ex)[:300])
else:
    print('Skipped')

dq_prof_df = collect('dq_profile_results', prof_results)
collect('dq_column_statistics', col_stats)

if not dq_prof_df.empty:
    display(dq_prof_df.groupby('Dimension').agg(
        Rules=('Rule', 'size'), Failing=('FailedRows', lambda s: int((s > 0).sum()))))
    display(dq_prof_df[(dq_prof_df['FailedRows'] > 0) & (dq_prof_df['Severity'] == 'HIGH')]
            .sort_values('FailedRows', ascending=False).head(30))


## 10 · Study / Site / Subject reconciliation
### `05_identifier_reconciliation` · Session S02

Compares the same identifier across every source that holds it and reports where they disagree.

| Output | Answers |
|---|---|
| Coverage summary | How many identifiers each source holds |
| Pairwise agreement | For each pair: in both, only A, only B, agreement % |
| Orphans | Present in one source and not another — the exception queue |
| Format profile | Length and shape variance per source |

The format profile matters more than it looks. `SITE-001`, `Site001` and `1` are frequently the same site, and a naive comparison reports a total mismatch that is really a normalisation problem. `RECON_NORMALISE` controls trimming, casing and leading zeros so you can separate a genuine gap from a cosmetic one — and avoid burning credibility on a false finding.

> This is the strongest **accuracy** evidence available without a gold standard. Two independent systems agreeing means something that one system being internally consistent does not.


In [ ]:
def normalise(col):
    c = col.cast('string')
    if RECON_NORMALISE.get('trim', True):
        c = F.trim(c)
    if RECON_NORMALISE.get('upper', True):
        c = F.upper(c)
    if RECON_NORMALISE.get('strip_leading_zeros', False):
        c = F.regexp_replace(c, r'^0+', '')
    return c


recon_summary, recon_pairs, recon_orphans, recon_format, recon_missing = [], [], [], [], []

if READ_DATA_CONTENT and RUN_RECONCILIATION:
    for entity, sources in IDENTIFIER_MAP.items():
        print('Entity: ' + entity)
        frames = {}
        for s in sources:
            df = read_table(s['table'])
            if df is None or s['column'] not in df.columns:
                recon_missing.append({'Entity': entity, 'Source': s['source'], 'Table': s['table'],
                                      'Column': s['column'], 'Issue': 'table or column not readable'})
                print('  --  ' + s['source'] + ' (' + s['table'] + ') not readable')
                continue
            raw = F.col('`' + s['column'] + '`')
            d = (df.select(raw.cast('string').alias('raw_value'), normalise(raw).alias('value'))
                   .filter(F.col('value').isNotNull() & (F.col('value') != '')).distinct())
            d.cache()
            frames[s['source']] = d
            n = d.count()
            recon_summary.append({'Entity': entity, 'Source': s['source'], 'Table': s['table'],
                                  'Column': s['column'], 'DistinctIdentifiers': n})
            print('  ok  ' + s['source'] + ': ' + str(n) + ' distinct')

            for r in (d.select(F.length('value').alias('len'),
                               F.when(F.col('value').rlike(r'^[0-9]+$'), 'numeric')
                                .when(F.col('value').rlike(r'^[A-Z]+$'), 'alpha')
                                .otherwise('mixed').alias('shape'))
                        .groupBy('len', 'shape').count()
                        .orderBy(F.desc('count')).limit(10).collect()):
                recon_format.append({'Entity': entity, 'Source': s['source'], 'Length': r['len'],
                                     'Shape': r['shape'], 'Count': r['count']})

        names = list(frames.keys())
        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                a, b = names[i], names[j]
                both = frames[a].join(frames[b].select('value'), 'value', 'inner').count()
                only_a = frames[a].join(frames[b].select('value'), 'value', 'left_anti').count()
                only_b = frames[b].join(frames[a].select('value'), 'value', 'left_anti').count()
                union = both + only_a + only_b
                recon_pairs.append({'Entity': entity, 'SourceA': a, 'SourceB': b, 'InBoth': both,
                                    'OnlyInA': only_a, 'OnlyInB': only_b,
                                    'AgreementPct': round(100.0 * both / union, 2) if union else None})
                print('    ' + a + ' vs ' + b + ': both=' + str(both)
                      + ' onlyA=' + str(only_a) + ' onlyB=' + str(only_b))

        if len(names) > 1:
            unioned = None
            for src, d in frames.items():
                tagged = d.select('value', 'raw_value', F.lit(src).alias('source'))
                unioned = tagged if unioned is None else unioned.unionByName(tagged)
            presence = (unioned.groupBy('value')
                               .agg(F.collect_set('source').alias('sources'),
                                    F.countDistinct('source').alias('source_count'),
                                    F.collect_set('raw_value').alias('raw_variants')))
            orphans = presence.filter(F.col('source_count') < len(names)).limit(5000).collect()
            for r in orphans:
                recon_orphans.append({'Entity': entity, 'Value': r['value'],
                                      'PresentIn': ', '.join(sorted(r['sources'])),
                                      'MissingFrom': ', '.join(sorted(set(names) - set(r['sources']))),
                                      'SourceCount': r['source_count'],
                                      'RawVariants': ', '.join(sorted(r['raw_variants'])[:5])})
            print('    orphans (not in every source): ' + str(len(orphans)))
else:
    print('Skipped')

collect('dq_recon_source_summary', recon_summary)
recon_pairs_df = collect('dq_recon_pairwise', recon_pairs)
recon_orph_df = collect('dq_recon_orphans', recon_orphans)
collect('dq_recon_format_profile', recon_format)
collect('dq_recon_unreadable', recon_missing)

if not recon_pairs_df.empty:
    display(recon_pairs_df.sort_values('AgreementPct'))
if not recon_orph_df.empty:
    display(recon_orph_df.groupby(['Entity', 'MissingFrom']).size()
            .to_frame('Orphans').sort_values('Orphans', ascending=False))


## 10b · Data reconciliation — layer-to-layer counts

Section 10 reconciles identifiers **across source systems**. This section reconciles them **down the medallion** — Bronze → Silver → Gold — and quantifies exactly where records are lost.

For each configured entity chain it produces:

| Output | Answers |
|---|---|
| **Layer counts** | Rows, distinct keys and duplicate keys at every layer |
| **Layer variance** | Between consecutive layers: how many keys were lost, how many appeared, net and percentage |
| **Chain summary** | End-to-end retention from the first layer to the last |
| **Group counts** | The same, broken down by study — so you see *which* study leaks |
| **Key exceptions** | The actual identifiers that dropped out, sampled |

Loss between layers is not automatically a defect. Screen failures legitimately do not reach Gold; a filter may be intentional. **The finding is unexplained variance, not variance.** That is why the output reports counts and lets the room supply the explanation, rather than labelling every difference an error.

> The per-study breakdown is the part that earns its place. A 2% overall loss looks tolerable until you see it is 100% of one study. Aggregate reconciliation hides exactly the failure mode you most need to find.

Configure `RECON_CHAINS` at the top of the next cell with real table and column names from the Part One output.


In [ ]:
# ---- CONFIGURE: replace with real table and column names -------------------
RECON_CHAINS = [
    {'entity': 'Subject', 'group_by': 'study_id', 'layers': [
        {'layer': 'Bronze', 'table': 'brz_edc_subjects',   'key': 'usubjid'},
        {'layer': 'Silver', 'table': 'slv_subject',        'key': 'usubjid'},
        {'layer': 'Gold',   'table': 'dim_subject',        'key': 'usubjid'}]},
    {'entity': 'AdverseEvent', 'group_by': 'study_id', 'layers': [
        {'layer': 'Bronze', 'table': 'brz_edc_ae',         'key': 'ae_uid'},
        {'layer': 'Silver', 'table': 'slv_adverse_event',  'key': 'ae_uid'},
        {'layer': 'Gold',   'table': 'fct_adverse_event',  'key': 'ae_uid'}]},
    {'entity': 'Site', 'group_by': 'country', 'layers': [
        {'layer': 'Bronze', 'table': 'brz_ucv_site',       'key': 'site_number'},
        {'layer': 'Silver', 'table': 'slv_site',           'key': 'site_id'},
        {'layer': 'Gold',   'table': 'dim_site',           'key': 'site_id'}]},
]
RECON_EXCEPTION_SAMPLE = 500
RECON_TOLERANCE_PCT = 0.0     # variance above this is reported as a break

layer_counts, layer_var, chain_summary, group_counts, key_exceptions = [], [], [], [], []


def key_set(df, key):
    return (df.select(F.col('`' + key + '`').cast('string').alias('k'))
              .filter(F.col('k').isNotNull() & (F.trim(F.col('k')) != ''))
              .distinct())


if READ_DATA_CONTENT and RUN_RECONCILIATION:
    for chain in RECON_CHAINS:
        entity, gb = chain['entity'], chain.get('group_by')
        print('Chain: ' + entity)
        resolved = []

        for spec in chain['layers']:
            df = read_table(spec['table'])
            if df is None or spec['key'] not in df.columns:
                layer_counts.append({'Entity': entity, 'Layer': spec['layer'], 'Table': spec['table'],
                                     'Key': spec['key'], 'RowCount': None, 'DistinctKeys': None,
                                     'DuplicateKeys': None, 'Status': 'TABLE_OR_KEY_NOT_FOUND'})
                print('  --  ' + spec['layer'] + ' ' + spec['table'] + ' not readable')
                continue
            ks = key_set(df, spec['key'])
            ks.cache()
            rows_n, keys_n = df.count(), ks.count()
            layer_counts.append({'Entity': entity, 'Layer': spec['layer'], 'Table': spec['table'],
                                 'Key': spec['key'], 'RowCount': rows_n, 'DistinctKeys': keys_n,
                                 'DuplicateKeys': max(rows_n - keys_n, 0), 'Status': 'OK'})
            resolved.append({'layer': spec['layer'], 'table': spec['table'], 'key': spec['key'],
                             'df': df, 'keys': ks, 'rows': rows_n, 'distinct': keys_n})
            print('  ok  ' + spec['layer'] + ': ' + str(rows_n) + ' rows, ' + str(keys_n) + ' distinct')

            if gb and gb in df.columns:
                for r in (df.groupBy(F.col('`' + gb + '`').cast('string').alias('g'))
                            .agg(F.countDistinct(F.col('`' + spec['key'] + '`')).alias('keys'),
                                 F.count(F.lit(1)).alias('rows'))
                            .limit(2000).collect()):
                    group_counts.append({'Entity': entity, 'GroupBy': gb, 'GroupValue': r['g'],
                                         'Layer': spec['layer'], 'DistinctKeys': r['keys'],
                                         'RowCount': r['rows']})

        for i in range(len(resolved) - 1):
            a, b = resolved[i], resolved[i + 1]
            lost_df = a['keys'].join(b['keys'], 'k', 'left_anti')
            gained_df = b['keys'].join(a['keys'], 'k', 'left_anti')
            lost, gained = lost_df.count(), gained_df.count()
            net = b['distinct'] - a['distinct']
            var_pct = round(100.0 * net / a['distinct'], 3) if a['distinct'] else None
            retention = round(100.0 * (a['distinct'] - lost) / a['distinct'], 3) if a['distinct'] else None
            if lost == 0 and gained == 0:
                status = 'RECONCILED'
            elif lost and gained:
                status = 'LOSS_AND_GAIN'
            elif lost:
                status = 'LOSS'
            else:
                status = 'GAIN'
            layer_var.append({'Entity': entity, 'FromLayer': a['layer'], 'ToLayer': b['layer'],
                              'FromTable': a['table'], 'ToTable': b['table'],
                              'FromDistinct': a['distinct'], 'ToDistinct': b['distinct'],
                              'KeysLost': lost, 'KeysGained': gained, 'NetVariance': net,
                              'VariancePct': var_pct, 'RetentionPct': retention, 'Status': status})
            print('    ' + a['layer'] + ' -> ' + b['layer'] + ': lost=' + str(lost)
                  + ' gained=' + str(gained) + ' net=' + str(net) + '  ' + status)

            for r in lost_df.limit(RECON_EXCEPTION_SAMPLE).collect():
                key_exceptions.append({'Entity': entity, 'Direction': 'LOST',
                                       'FromLayer': a['layer'], 'ToLayer': b['layer'],
                                       'KeyValue': r['k']})
            for r in gained_df.limit(RECON_EXCEPTION_SAMPLE).collect():
                key_exceptions.append({'Entity': entity, 'Direction': 'GAINED',
                                       'FromLayer': a['layer'], 'ToLayer': b['layer'],
                                       'KeyValue': r['k']})

        if len(resolved) >= 2:
            first, last = resolved[0], resolved[-1]
            e2e_lost = first['keys'].join(last['keys'], 'k', 'left_anti').count()
            chain_summary.append({
                'Entity': entity,
                'FirstLayer': first['layer'], 'FirstTable': first['table'], 'FirstDistinct': first['distinct'],
                'LastLayer': last['layer'], 'LastTable': last['table'], 'LastDistinct': last['distinct'],
                'KeysLostEndToEnd': e2e_lost,
                'EndToEndRetentionPct': round(100.0 * (first['distinct'] - e2e_lost) / first['distinct'], 3)
                if first['distinct'] else None,
                'Reconciled': e2e_lost == 0})
else:
    print('Skipped')

recon_layer_df = collect('dq_recon_layer_counts', layer_counts)
recon_var_df = collect('dq_recon_layer_variance', layer_var)
recon_chain_df = collect('dq_recon_chain_summary', chain_summary)
recon_group_df = collect('dq_recon_group_counts', group_counts)
collect('dq_recon_key_exceptions', key_exceptions)

# per-group retention: which study is actually leaking
group_var = []
if not recon_group_df.empty:
    for (ent, gval), grp in recon_group_df.groupby(['Entity', 'GroupValue']):
        chain = next((c for c in RECON_CHAINS if c['entity'] == ent), None)
        if not chain:
            continue
        order = [l['layer'] for l in chain['layers']]
        present = grp.set_index('Layer')['DistinctKeys'].to_dict()
        first_layer = next((l for l in order if l in present), None)
        last_layer = next((l for l in reversed(order) if l in present), None)
        if not first_layer or first_layer == last_layer:
            continue
        f, t = present[first_layer], present[last_layer]
        group_var.append({'Entity': ent, 'GroupValue': gval, 'FirstLayer': first_layer,
                          'FirstKeys': f, 'LastLayer': last_layer, 'LastKeys': t,
                          'NetVariance': t - f,
                          'RetentionPct': round(100.0 * t / f, 2) if f else None})
group_var_df = collect('dq_recon_group_variance', group_var)

if not recon_chain_df.empty:
    display(recon_chain_df)
if not recon_var_df.empty:
    breaks = recon_var_df[recon_var_df['Status'] != 'RECONCILED']
    if not breaks.empty:
        display(breaks.sort_values('KeysLost', ascending=False))
if not group_var_df.empty:
    worst = group_var_df[group_var_df['RetentionPct'].notna()].sort_values('RetentionPct').head(20)
    if not worst.empty:
        print()
        print('Worst-retaining groups - aggregate reconciliation hides these:')
        display(worst)


## 11 · Great Expectations validation suite
### `10_great_expectations` · Session S03

Translates `RULES` into a Great Expectations suite and runs it.

The **suite JSON is always generated** — it is the durable artefact: version-controllable, reviewable, and the thing that becomes a blocking gate in a pipeline later. Execution takes one of two paths:

| Engine | When |
|---|---|
| `gx` | `great_expectations` is importable — you published a custom Fabric Environment containing it |
| `native` | Otherwise. Equivalent checks in PySpark, labelled honestly. |

The native fallback exists because publishing a Fabric Environment is a change-controlled action at a regulated organisation and should not block a discovery workshop.

Generated: `expect_column_values_to_not_be_null` · `..._to_be_in_set` · `..._to_be_between` · `..._to_match_regex` · `expect_compound_columns_to_be_unique` · `expect_table_row_count_to_be_between`

Written to `Files/expectations/` for source control.


In [ ]:
def build_suite(table, rules):
    exps = []

    def add(etype, kwargs, note):
        exps.append({'expectation_type': etype, 'kwargs': kwargs,
                     'meta': {'notes': note, 'generated_by': 'Fabric-Estate-Scan'}})

    for c in (rules.get('not_null') or []):
        add('expect_column_values_to_not_be_null', {'column': c}, 'declared mandatory')
    for c, allowed in (rules.get('allowed_values') or {}).items():
        add('expect_column_values_to_be_in_set', {'column': c, 'value_set': list(allowed)}, 'controlled terminology')
    for c, rng in (rules.get('ranges') or {}).items():
        add('expect_column_values_to_be_between',
            {'column': c, 'min_value': rng[0], 'max_value': rng[1]}, 'plausible range')
    for c, pat in (rules.get('patterns') or {}).items():
        add('expect_column_values_to_match_regex', {'column': c, 'regex': pat}, 'identifier format')
    if rules.get('business_key'):
        add('expect_compound_columns_to_be_unique', {'column_list': list(rules['business_key'])}, 'business key')
    if rules.get('expected_row_range'):
        rr = rules['expected_row_range']
        add('expect_table_row_count_to_be_between',
            {'min_value': rr[0], 'max_value': rr[1]}, 'volume plausibility')
    return {'expectation_suite_name': table + '_suite', 'data_asset_type': 'Dataset',
            'meta': {'great_expectations_version': '0.18.x'}, 'expectations': exps}


def run_native(df, exp, total):
    et, kw = exp['expectation_type'], exp['kwargs']
    cols = df.columns
    if et == 'expect_table_row_count_to_be_between':
        return (0 if kw['min_value'] <= total <= kw['max_value'] else 1), 1
    if et == 'expect_compound_columns_to_be_unique':
        keys = kw['column_list']
        if not all(k in cols for k in keys):
            return None, None
        dup = (df.groupBy(*[F.col('`' + k + '`') for k in keys]).count()
                 .filter(F.col('count') > 1)
                 .agg(F.coalesce(F.sum(F.col('count') - 1), F.lit(0))).collect()[0][0])
        return int(dup or 0), total
    c = kw.get('column')
    if c not in cols:
        return None, None
    col = F.col('`' + c + '`')
    if et == 'expect_column_values_to_not_be_null':
        return df.filter(col.isNull() | (F.trim(col.cast('string')) == '')).count(), total
    if et == 'expect_column_values_to_be_in_set':
        allowed = [str(v).upper() for v in kw['value_set']]
        return df.filter(col.isNotNull() & ~F.upper(F.trim(col.cast('string'))).isin(allowed)).count(), total
    if et == 'expect_column_values_to_be_between':
        return df.filter(col.isNotNull() & ~col.between(kw['min_value'], kw['max_value'])).count(), total
    if et == 'expect_column_values_to_match_regex':
        return df.filter(col.isNotNull() & ~col.cast('string').rlike(kw['regex'])).count(), total
    return None, None


GX_AVAILABLE = False
if READ_DATA_CONTENT and RUN_EXPECTATIONS:
    try:
        import great_expectations as gx  # noqa: F401
        GX_AVAILABLE = True
        print('great_expectations available')
    except ImportError:
        print('great_expectations not installed - using the native engine')
        print('  to enable GX: create a Fabric Environment, add the great_expectations')
        print('  package, publish it, and attach it to this notebook')

suites, val_rows = {}, []

if READ_DATA_CONTENT and RUN_EXPECTATIONS:
    targets = [t for t in DATA_TABLES if t in RULES] or list(RULES.keys())
    for t in targets:
        suite = build_suite(t, RULES.get(t, {}))
        suites[t] = suite
        df = read_table(t)
        if df is None:
            val_rows.append({'Table': t, 'ExpectationType': 'table_readable', 'Column': t,
                             'Success': False, 'UnexpectedCount': None, 'ElementCount': None,
                             'Engine': 'native', 'Note': 'table not readable'})
            continue
        total = df.count()
        for exp in suite['expectations']:
            failed, elems = run_native(df, exp, total)
            colname = str(exp['kwargs'].get('column') or exp['kwargs'].get('column_list'))
            if failed is None:
                val_rows.append({'Table': t, 'ExpectationType': exp['expectation_type'], 'Column': colname,
                                 'Success': None, 'UnexpectedCount': None, 'ElementCount': total,
                                 'Engine': 'native', 'Note': 'column not present - not evaluated'})
                continue
            val_rows.append({'Table': t, 'ExpectationType': exp['expectation_type'], 'Column': colname,
                             'Success': failed == 0, 'UnexpectedCount': failed, 'ElementCount': elems,
                             'UnexpectedPct': round(100.0 * failed / elems, 4) if elems else None,
                             'Engine': 'gx' if GX_AVAILABLE else 'native', 'Note': exp['meta']['notes']})
        write_file('expectations/' + t + '_suite.json', json.dumps(suite, indent=2))
    if suites:
        write_file('expectations/_all_suites.json', json.dumps(suites, indent=2))
else:
    print('Skipped')

dq_val_df = collect('dq_expectation_results', val_rows)
collect('dq_expectation_suites', [{'Table': k, 'ExpectationCount': len(v['expectations']),
                                   'SuiteJson': json.dumps(v)[:8000]} for k, v in suites.items()])

if not dq_val_df.empty:
    display(dq_val_df.groupby(['Engine', 'Success']).size().to_frame('Count'))
    display(dq_val_df[dq_val_df['Success'] == False].sort_values('UnexpectedCount', ascending=False).head(30))


## 12 · Candidate semantic model (TMDL)
### `08_semantic_candidate` · Session S05

Reads the Gold layer, classifies tables as facts or dimensions, infers relationships from column-name matches against dimension keys, and emits **TMDL** to `Files/tmdl/`.

> A **candidate**, not a deployable model. Inferred relationships encode what the column names imply, which is not always what the business means. Each carries a confidence score and the reason it was inferred, so the review is about accepting or rejecting specific proposals rather than reading TMDL cold.

Cardinality is inferred by testing whether the dimension-side column is *actually* unique. Where it is not, the relationship is emitted as many-to-many and flagged — a many-to-many on what looks like a dimension key is usually a modelling defect worth raising, not a modelling choice.

Needs only schema and row counts, so it runs with lighter data access than sections 9–11.


In [ ]:
TMDL_TYPES = {'string': 'string', 'boolean': 'boolean', 'date': 'dateTime', 'timestamp': 'dateTime',
              'int': 'int64', 'bigint': 'int64', 'smallint': 'int64', 'tinyint': 'int64',
              'double': 'double', 'float': 'double'}


def tmdl_type(spark_type):
    base = str(spark_type).split('(')[0]
    return TMDL_TYPES.get(base, 'decimal' if base.startswith('decimal') else 'string')


def classify(name):
    n = name.lower()
    if n.startswith('fct_') or n.startswith('fact_') or n.endswith('_fact'):
        return 'fact'
    if n.startswith('dim_') or n.endswith('_dim'):
        return 'dimension'
    return 'unknown'


gold_tables, tmdl_cols, tmdl_rels, tmdl_files = [], [], [], {}

if READ_DATA_CONTENT and RUN_SEMANTIC_MODEL:
    candidates = [t for t in DATA_TABLES if t.lower().startswith(GOLD_PREFIXES)]
    if not candidates:
        candidates = [t for t in DATA_TABLES if classify(t) != 'unknown']
    print(str(len(candidates)) + ' Gold candidates')

    schemas = {}
    for t in candidates:
        df = read_table(t)
        if df is None:
            continue
        schemas[t] = [(f.name, f.dataType.simpleString()) for f in df.schema.fields]
        gold_tables.append({'Table': t, 'Role': classify(t), 'Columns': len(schemas[t]),
                            'RowCount': df.count()})
        for c, ty in schemas[t]:
            tmdl_cols.append({'Table': t, 'Column': c, 'SparkType': ty, 'TmdlType': tmdl_type(ty),
                              'IsKeyCandidate': c.lower().endswith('_id') or c.lower().endswith('_key')})

    # a dimension's key is the *_id / *_key column that is actually unique
    dim_keys = {}
    for t, cols in schemas.items():
        if classify(t) != 'dimension':
            continue
        df = read_table(t)
        total = df.count() if df is not None else 0
        for c, _ in cols:
            if c.lower().endswith('_id') or c.lower().endswith('_key'):
                distinct = df.select(F.col('`' + c + '`')).distinct().count() if df is not None else 0
                if total and distinct == total:
                    dim_keys[t] = (c, True)
                    break
                dim_keys.setdefault(t, (c, False))

    for fact, cols in schemas.items():
        if classify(fact) != 'fact':
            continue
        fact_cols = {c.lower(): c for c, _ in cols}
        for dim, (dkey, is_unique) in dim_keys.items():
            if dkey.lower() in fact_cols:
                tmdl_rels.append({
                    'FromTable': fact, 'FromColumn': fact_cols[dkey.lower()],
                    'ToTable': dim, 'ToColumn': dkey,
                    'ToCardinality': 'one' if is_unique else 'many',
                    'CrossFilter': 'oneDirection',
                    'Confidence': 'HIGH' if is_unique else 'LOW',
                    'Reason': 'column name match on dimension key'
                              + ('' if is_unique else '; dimension key is NOT unique - review')})

    for t, cols in schemas.items():
        lines = ['table ' + t, '\tlineageTag: ' + str(uuid.uuid4()), '']
        for c, ty in cols:
            summarize = 'sum' if (tmdl_type(ty) in ('int64', 'double', 'decimal')
                                  and classify(t) == 'fact') else 'none'
            lines += ['\tcolumn ' + c, '\t\tdataType: ' + tmdl_type(ty),
                      '\t\tlineageTag: ' + str(uuid.uuid4()),
                      '\t\tsummarizeBy: ' + summarize, '\t\tsourceColumn: ' + c, '']
        lines += ['\tpartition ' + t + ' = entity', '\t\tmode: directLake', '\t\tsource',
                  '\t\t\tentityName: ' + t, '\t\t\texpressionSource: DatabaseQuery', '']
        tmdl_files[t + '.tmdl'] = '\n'.join(lines)

    rel_lines = []
    for r in tmdl_rels:
        rel_lines += ['relationship ' + str(uuid.uuid4()),
                      '\tfromColumn: ' + r['FromTable'] + '.' + r['FromColumn'],
                      '\ttoColumn: ' + r['ToTable'] + '.' + r['ToColumn'],
                      '\ttoCardinality: ' + r['ToCardinality'],
                      '\tcrossFilteringBehavior: ' + r['CrossFilter'],
                      '\t// confidence: ' + r['Confidence'] + ' - ' + r['Reason'], '']
    tmdl_files['relationships.tmdl'] = '\n'.join(rel_lines)

    model_lines = ['model Model', '\tculture: en-GB', '\tdefaultPowerBIDataSourceVersion: powerBI_V3',
                   '\tsourceQueryCulture: en-GB', '',
                   '\t// Candidate generated from the Gold layer.',
                   '\t// Relationships are inferred proposals - review before deployment.', '']
    for t in schemas:
        model_lines.append('\tref table ' + t)
    tmdl_files['model.tmdl'] = '\n'.join(model_lines)

    for fname, content in tmdl_files.items():
        write_file('tmdl/' + fname, content)
else:
    print('Skipped')

collect('dq_semantic_candidate_tables', gold_tables)
collect('dq_semantic_candidate_columns', tmdl_cols)
tmdl_rels_df = collect('dq_semantic_candidate_relationships', tmdl_rels)
collect('dq_semantic_candidate_tmdl', [{'FileName': k, 'Content': v} for k, v in tmdl_files.items()])

if not tmdl_rels_df.empty:
    display(tmdl_rels_df.sort_values('Confidence'))
    low = tmdl_rels_df[tmdl_rels_df['Confidence'] == 'LOW']
    if not low.empty:
        print()
        print(str(len(low)) + ' relationship(s) point at a NON-UNIQUE dimension key.')
        print('That is usually a modelling defect, not a modelling choice. Raise it.')


## 13 · Fabric IQ ontology binding spec
### `09_ontology_binding` · Session S05

Maps clinical **concepts** — Study, Site, Subject, Visit, AdverseEvent — onto physical Gold tables and columns, using the relationships inferred in section 12. Written to `Files/ontology/` as YAML and JSON, plus a printed application checklist.

> Generated here, **applied by hand** in the Fabric portal. Fabric IQ Ontology is in preview with no stable programmatic binding API, so the spec is the deliverable and the portal work is manual. A deliberate boundary, not a limitation.

The concept layer is where an agent stops needing to know your table names. `CONCEPT_MAP` controls the mapping; anything unmapped is named after the table and flagged for naming review — an ontology named after your tables teaches agents your schema rather than your business.


In [ ]:
def to_yaml(obj, indent=0):
    pad = '  ' * indent
    out = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, (dict, list)) and v:
                out.append(pad + str(k) + ':')
                out.append(to_yaml(v, indent + 1))
            elif isinstance(v, (dict, list)):
                out.append(pad + str(k) + (': {}' if isinstance(v, dict) else ': []'))
            else:
                val = '' if v is None else str(v)
                if val == '' or any(ch in val for ch in ':#{}[],&*?|<>=!%@`'):
                    val = '"' + val.replace('"', "'") + '"'
                out.append(pad + str(k) + ': ' + val)
    elif isinstance(obj, list):
        for item in obj:
            if isinstance(item, (dict, list)):
                block = to_yaml(item, indent + 1)
                first, rest = block.split('\n', 1) if '\n' in block else (block, '')
                out.append(pad + '- ' + first.strip())
                if rest:
                    out.append(rest)
            else:
                out.append(pad + '- ' + str(item))
    return '\n'.join(out)


ontology, concept_rows, binding_rows = {}, [], []

if READ_DATA_CONTENT and RUN_ONTOLOGY:
    tabs = COLLECTED.get('dq_semantic_candidate_tables', pd.DataFrame())
    cols_df = COLLECTED.get('dq_semantic_candidate_columns', pd.DataFrame())
    rels = COLLECTED.get('dq_semantic_candidate_relationships', pd.DataFrame())

    if tabs.empty:
        print('No Gold tables available - run section 12 first')
    else:
        concepts = []
        for _, t in tabs.iterrows():
            tname = t['Table']
            cname = CONCEPT_MAP.get(tname)
            inferred = cname is None
            if inferred:
                cname = ''.join(w.capitalize()
                                for w in re.sub(r'^(fct_|dim_|gold_)', '', tname).split('_'))

            tcols = cols_df[cols_df['Table'] == tname] if not cols_df.empty else pd.DataFrame()
            key = None
            for _, c in tcols.iterrows():
                low = str(c['Column']).lower()
                if low.endswith('_id') or low.endswith('_key'):
                    key = c['Column']
                    break

            props = [{'name': c['Column'], 'dataType': c['TmdlType'], 'boundColumn': c['Column'],
                      'role': 'identifier' if c['Column'] == key else 'attribute'}
                     for _, c in tcols.iterrows()]

            rel_out = []
            if not rels.empty:
                for _, r in rels[rels['FromTable'] == tname].iterrows():
                    target = CONCEPT_MAP.get(r['ToTable'], r['ToTable'])
                    rel_out.append({'name': 'has' + str(target), 'targetConcept': target,
                                    'fromColumn': r['FromColumn'], 'toColumn': r['ToColumn'],
                                    'cardinality': 'many-to-one' if r['ToCardinality'] == 'one' else 'many-to-many',
                                    'confidence': r['Confidence']})

            concepts.append({'name': cname, 'sourceTable': tname, 'role': t['Role'],
                             'identifier': key, 'rowCount': int(t['RowCount']),
                             'binding': {'lakehouse': 'attached', 'table': tname, 'keyColumn': key},
                             'properties': props, 'relationships': rel_out,
                             'namingReviewRequired': inferred})
            concept_rows.append({'Concept': cname, 'SourceTable': tname, 'Role': t['Role'],
                                 'Identifier': key, 'Properties': len(props),
                                 'Relationships': len(rel_out), 'NamingInferred': inferred})
            for p in props:
                binding_rows.append({'Concept': cname, 'Property': p['name'], 'BoundTable': tname,
                                     'BoundColumn': p['boundColumn'], 'DataType': p['dataType'],
                                     'Role': p['role']})

        ontology = {'ontology': ONTOLOGY_NAME, 'version': '0.1-candidate',
                    'generatedUtc': SCAN_START,
                    'status': 'CANDIDATE - review before applying in the Fabric portal',
                    'conceptCount': len(concepts), 'concepts': concepts}

        write_file('ontology/' + ONTOLOGY_NAME + '.yaml', to_yaml(ontology))
        write_file('ontology/' + ONTOLOGY_NAME + '.json', json.dumps(ontology, indent=2, default=str))

        print()
        print('APPLICATION CHECKLIST - Fabric IQ Ontology (preview), applied by hand')
        print('  1. Confirm Fabric IQ Ontology is enabled on the tenant')
        print('  2. Create the ontology named: ' + ONTOLOGY_NAME)
        print('  3. Create one concept per row in estate_dq_ontology_concepts')
        print('  4. Bind each concept to its lakehouse table and key column')
        print('  5. Add properties from estate_dq_ontology_bindings')
        print('  6. Add relationships - reject any marked confidence LOW until reviewed')
        print('  7. Rename any concept flagged NamingInferred with the business term')
else:
    print('Skipped')

dq_conc_df = collect('dq_ontology_concepts', concept_rows)
collect('dq_ontology_bindings', binding_rows)
collect('dq_ontology_spec',
        [{'Name': ONTOLOGY_NAME, 'Format': 'yaml', 'Content': to_yaml(ontology)[:60000]}] if ontology else [])

if not dq_conc_df.empty:
    display(dq_conc_df)
    unnamed = dq_conc_df[dq_conc_df['NamingInferred'] == True]
    if not unnamed.empty:
        print()
        print(str(len(unnamed)) + ' concept name(s) were inferred from table names.')
        print('An ontology named after your tables teaches agents your schema, not your business.')


In [ ]:
if READ_DATA_CONTENT:
    if not dq_prof_df.empty:
        for _, r in dq_prof_df[(dq_prof_df['FailedRows'] > 0)
                               & (dq_prof_df['Severity'] == 'HIGH')].iterrows():
            finding('HIGH', 'Data quality: ' + str(r['Dimension']),
                    str(r['Rule']) + ' failed on ' + str(r['Scope']) + ' - '
                    + str(r['FailedRows']) + ' of ' + str(r['TotalRows']) + ' rows',
                    str(r['Table']) + ' | pass rate ' + str(r['PassRate']) + '%',
                    'Is this a source problem, a transformation problem, or an expectation nobody agreed?')

    if not recon_pairs_df.empty:
        for _, r in recon_pairs_df[recon_pairs_df['AgreementPct'].notna()].iterrows():
            if r['AgreementPct'] < 95:
                finding('HIGH', 'Identifier reconciliation',
                        str(r['Entity']) + ': ' + str(r['SourceA']) + ' and ' + str(r['SourceB'])
                        + ' agree on only ' + str(r['AgreementPct']) + '% of identifiers',
                        'onlyA=' + str(r['OnlyInA']) + ' onlyB=' + str(r['OnlyInB'])
                        + ' both=' + str(r['InBoth']),
                        'Is this a genuine gap or a formatting difference? Check the format profile first.')

    # ---- layer-to-layer reconciliation ------------------------------------
    if not recon_chain_df.empty:
        for _, c in recon_chain_df[recon_chain_df['Reconciled'] == False].iterrows():
            finding('HIGH', 'Layer reconciliation',
                    str(c['Entity']) + ': ' + str(c['KeysLostEndToEnd']) + ' keys do not survive '
                    + str(c['FirstLayer']) + ' to ' + str(c['LastLayer'])
                    + ' (' + str(c['EndToEndRetentionPct']) + '% retained)',
                    str(c['FirstTable']) + ' -> ' + str(c['LastTable']),
                    'Is that loss intended - screen failures, filters - or unexplained? Who can confirm?')

    if not recon_var_df.empty:
        for _, v in recon_var_df[recon_var_df['KeysLost'] > 0].iterrows():
            finding('HIGH', 'Layer reconciliation',
                    str(v['Entity']) + ': ' + str(v['KeysLost']) + ' keys lost between '
                    + str(v['FromLayer']) + ' and ' + str(v['ToLayer']),
                    str(v['FromTable']) + ' (' + str(v['FromDistinct']) + ') -> '
                    + str(v['ToTable']) + ' (' + str(v['ToDistinct']) + ')',
                    'Show us the rule that removes them, or the exception queue that holds them.')
        for _, v in recon_var_df[recon_var_df['KeysGained'] > 0].iterrows():
            finding('MEDIUM', 'Layer reconciliation',
                    str(v['Entity']) + ': ' + str(v['KeysGained']) + ' keys appear at '
                    + str(v['ToLayer']) + ' that are not in ' + str(v['FromLayer']),
                    str(v['ToTable']),
                    'Where do these come from - another source, or generated in transformation?')

    if not recon_layer_df.empty:
        dups = recon_layer_df[(recon_layer_df['DuplicateKeys'].notna())
                              & (recon_layer_df['DuplicateKeys'] > 0)]
        for _, d in dups.iterrows():
            finding('HIGH', 'Layer reconciliation',
                    str(d['Entity']) + ' at ' + str(d['Layer']) + ': ' + str(d['DuplicateKeys'])
                    + ' duplicate keys on ' + str(d['Key']),
                    str(d['Table']) + ' | ' + str(d['RowCount']) + ' rows, '
                    + str(d['DistinctKeys']) + ' distinct',
                    'Is the declared key actually the grain of this table?')
        missing = recon_layer_df[recon_layer_df['Status'] != 'OK']
        if len(missing):
            finding('MEDIUM', 'Evidence gap',
                    str(len(missing)) + ' reconciliation layer(s) unreadable',
                    ', '.join((missing['Entity'] + '/' + missing['Layer']).astype(str).tolist()[:10]),
                    'Are these the right table names? Reconciliation is incomplete without them.')

    if not group_var_df.empty:
        for _, g in group_var_df[(group_var_df['RetentionPct'].notna())
                                 & (group_var_df['RetentionPct'] < 90)].sort_values('RetentionPct').head(15).iterrows():
            finding('HIGH', 'Layer reconciliation by group',
                    str(g['Entity']) + ' for ' + str(g['GroupValue']) + ': only '
                    + str(g['RetentionPct']) + '% retained ' + str(g['FirstLayer'])
                    + ' to ' + str(g['LastLayer']),
                    str(g['FirstKeys']) + ' -> ' + str(g['LastKeys']),
                    'What is different about this one? Aggregate reconciliation hid this.')

    if not dq_val_df.empty:
        failed = dq_val_df[dq_val_df['Success'] == False]
        if len(failed):
            finding('MEDIUM', 'Validation suite',
                    str(len(failed)) + ' expectation(s) failed across '
                    + str(failed['Table'].nunique()) + ' table(s)',
                    'engine=' + str(dq_val_df['Engine'].iloc[0]),
                    'Should these become blocking gates, and who owns the exceptions?')

    if not tmdl_rels_df.empty:
        for _, r in tmdl_rels_df[tmdl_rels_df['Confidence'] == 'LOW'].iterrows():
            finding('MEDIUM', 'Data model',
                    'Relationship ' + str(r['FromTable']) + ' -> ' + str(r['ToTable'])
                    + ' points at a non-unique dimension key',
                    str(r['ToTable']) + '.' + str(r['ToColumn']),
                    'Is this dimension actually a dimension, or a de-duplicated fact?')

    if not dq_conc_df.empty:
        inferred = dq_conc_df[dq_conc_df['NamingInferred'] == True]
        if len(inferred):
            finding('LOW', 'Ontology',
                    str(len(inferred)) + ' concept name(s) inferred from table names',
                    ', '.join(inferred['Concept'].astype(str).head(10).tolist()),
                    'What is the business term for each of these?')

find_df = pd.DataFrame(findings)
if not find_df.empty:
    order = {'HIGH': 0, 'MEDIUM': 1, 'LOW': 2}
    find_df = (find_df.assign(_o=find_df['Severity'].map(order))
                      .sort_values(['_o', 'Area']).drop(columns='_o'))
COLLECTED['findings'] = find_df

print('Findings after the data pass: ' + str(len(find_df)))
if not find_df.empty:
    display(find_df.groupby(['Severity', 'Area']).size().to_frame('Count'))


## 14 · Persist, summarise, and declare what you did not get

Writes every collected table to the attached Lakehouse — `estate_*` for metadata, `estate_dq_*` for data quality — so the run is queryable, comparable across runs and reportable in Power BI.

The **coverage gaps** block at the end is the important part. If the scanner was rejected, a SQL endpoint was unreachable, or `READ_DATA_CONTENT` was off, say so before you present anything. An incomplete scan presented as complete is how a discovery exercise loses the room.


In [ ]:
def save(name, df):
    if df is None or df.empty:
        return 0
    out = df.copy()
    # all-string columns: an all-null column breaks Spark schema inference
    for c in out.columns:
        out[c] = out[c].astype(str).replace({'None': '', 'nan': '', 'NaT': '', '<NA>': ''})
    out['ScanTimestampUtc'] = SCAN_START
    (spark.createDataFrame(out)
          .write.mode('overwrite').option('overwriteSchema', 'true')
          .saveAsTable(TABLE_PREFIX + name))
    return len(out)


if SAVE_TO_LAKEHOUSE:
    print('Writing to the attached lakehouse')
    for name, df in COLLECTED.items():
        n = save(name, df)
        if n:
            print('  ' + TABLE_PREFIX + name + ': ' + str(n))
else:
    print('SAVE_TO_LAKEHOUSE is False - results remain in memory only')


def count(name):
    return len(COLLECTED.get(name, pd.DataFrame()))


views = int((obj_df['object_kind'] == 'VIEW').sum()) if not obj_df.empty else 0
persisted = int((obj_df['object_kind'] == 'TABLE').sum()) if not obj_df.empty else 0
chained = int((chain_df['ChainDepth'] >= 2).sum()) if not chain_df.empty else 0
rebuilt = 0
if not delta_df.empty and 'MaintenancePattern' in delta_df.columns:
    rebuilt = int((delta_df['MaintenancePattern'] == 'FULL REBUILD (Overwrite)').sum())

lin_auth = lin_inferred = 0
if not lineage_df.empty:
    lin_auth = int((lineage_df['Confidence'] == 'HIGH').sum())
    lin_inferred = int((lineage_df['Confidence'] != 'HIGH').sum())
max_depth = int(paths_df['Depth'].max()) if not paths_df.empty else 0

worst_agreement = None
if not recon_pairs_df.empty and recon_pairs_df['AgreementPct'].notna().any():
    worst_agreement = float(recon_pairs_df['AgreementPct'].min())

chains_broken = keys_lost = 0
worst_retention = None
if not recon_chain_df.empty:
    chains_broken = int((recon_chain_df['Reconciled'] == False).sum())
    worst_retention = float(recon_chain_df['EndToEndRetentionPct'].min())
if not recon_var_df.empty:
    keys_lost = int(recon_var_df['KeysLost'].fillna(0).sum())

summary = {
    'ScanTimestampUtc': SCAN_START,
    'Scope': SCOPE,
    'ReadDataContent': READ_DATA_CONTENT,
    '--- PART ONE: METADATA ---': '',
    'Capacities': len(cap_df),
    'Workspaces': len(ws_df),
    'Items': len(items_df),
    'LakehouseTables': len(tables_df),
    'DeltaTablesProfiled': len(delta_df),
    'DeltaFullRebuildTables': rebuilt,
    'Shortcuts': len(sc_df),
    'ExternalShortcuts': int((sc_df['IsExternal'] == True).sum()) if not sc_df.empty else 0,
    'SemanticModels': count('semantic_models'),
    'ModelRelationships': count('model_relationships'),
    'RlsRules': count('rls_roles'),
    'SqlViews': views,
    'SqlPersistedTables': persisted,
    'ChainedViews': chained,
    'ThrottlingEvents': throttle_events,
    '--- LINEAGE ---': '',
    'LineageEdgesTotal': count('lineage_edges'),
    'LineageAuthoritative': lin_auth,
    'LineageInferred': lin_inferred,
    'ViewColumnLineageRows': count('lineage_column_view'),
    'EndToEndPaths': count('lineage_paths'),
    'LongestChainHops': max_depth,
    'GraphNodes': count('graph_nodes'),
    'GraphEdges': count('graph_edges'),
    '--- PART TWO: DATA ---': '',
    'TablesProfiled': len(DATA_TABLES),
    'RulesEvaluated': count('dq_profile_results'),
    'RulesFailing': int((dq_prof_df['FailedRows'] > 0).sum()) if not dq_prof_df.empty else 0,
    'ReconSourcePairs': count('dq_recon_pairwise'),
    'WorstCrossSourceAgreementPct': worst_agreement,
    'ReconOrphans': count('dq_recon_orphans'),
    '--- LAYER RECONCILIATION ---': '',
    'ReconChains': count('dq_recon_chain_summary'),
    'ReconChainsNotReconciled': chains_broken,
    'ReconKeysLostTotal': keys_lost,
    'WorstEndToEndRetentionPct': worst_retention,
    'ReconGroupsMeasured': count('dq_recon_group_variance'),
    'ReconKeyExceptionsSampled': count('dq_recon_key_exceptions'),
    '--- DESIGN OUTPUT ---': '',
    'ExpectationsRun': count('dq_expectation_results'),
    'ExpectationEngine': ('gx' if GX_AVAILABLE else 'native') if READ_DATA_CONTENT else 'not run',
    'RelationshipsProposed': count('dq_semantic_candidate_relationships'),
    'OntologyConcepts': count('dq_ontology_concepts'),
    'Findings': len(find_df),
}
display(pd.DataFrame([summary]).T.rename(columns={0: 'Value'}))

gaps = []
if not scan_results:
    gaps.append('Scanner API did not run - no relationships, RLS, lineage or model schema')
if count('sql_unreachable'):
    gaps.append(str(count('sql_unreachable')) + ' SQL endpoint(s) unreachable - view inventory is a floor, not a total')
if not INCLUDE_DEFINITIONS:
    gaps.append('Definitions skipped - no code lineage and no pipeline flow reconstruction')
if not PROFILE_DELTA:
    gaps.append('Delta profiling skipped - no rebuild-vs-MERGE evidence')
if not count('capacity_metrics'):
    gaps.append('Capacity Metrics not reachable - no throttling or CU evidence')
if lin_inferred:
    gaps.append(str(lin_inferred) + ' of ' + str(lin_auth + lin_inferred)
                + ' lineage edges are INFERRED from static code analysis, not observed execution')
gaps.append('Column-level lineage exists only for SQL views. It cannot be derived across Spark '
            'transformations from any Fabric API - that break in the chain is itself a finding.')
if not READ_DATA_CONTENT:
    gaps.append('PART TWO NOT RUN - no data quality, reconciliation, model or ontology output')
else:
    unread = count('dq_recon_layer_counts')
    if not recon_layer_df.empty:
        bad_layers = int((recon_layer_df['Status'] != 'OK').sum())
        if bad_layers:
            gaps.append(str(bad_layers) + ' reconciliation layer(s) unreadable - layer counts are partial')
    if chains_broken:
        gaps.append(str(chains_broken) + ' entity chain(s) do not reconcile end to end - '
                    'variance is reported, the EXPLANATION is not. Ask before calling it a defect.')
    if count('dq_recon_unreadable'):
        gaps.append(str(count('dq_recon_unreadable')) + ' identifier source(s) unreadable - cross-source reconciliation is partial')
    if not GX_AVAILABLE and RUN_EXPECTATIONS:
        gaps.append('Great Expectations not installed - suites generated, executed by the native engine')
    untyped = [t for t in DATA_TABLES if t not in RULES]
    if untyped:
        gaps.append(str(len(untyped)) + ' table(s) have no rules - only completeness, uniqueness and accuracy measured')

print()
print('COVERAGE GAPS - say these out loud before presenting anything:')
for g in gaps:
    print('  - ' + g)

print()
print('Delivery scripts covered: 01, 02, 03, 04, 05, 06, 07, 08, 09, 10')
print('Sessions: S01 (Part One), S02, S03, S05 (Part Two)')
print()
print('Findings are hypotheses to test in the session, not conclusions.')
print('Reconciliation reports variance. It does not know which variance is intended.')
print('HIGH-confidence lineage is the engine\'s own answer. MEDIUM is what the code says,')
print('not what executed. Present them differently.')
